## Système de reconnaissance faciale — Notebook unique, commenté (moteur IA + MongoDB + API Flask)

Ce notebook contient tout le pipeline : chargement des modèles InsightFace, communication avec MongoDB,
et l'API Flask qui sert le flux vidéo (MJPEG) et les pages du tableau de bord.

Chaque section est précédée d'une cellule Markdown expliquant : **le rôle** de la cellule, **son fonctionnement**,
les **principes mathématiques** sous-jacents quand il y en a, et l'**origine/histoire** des technologies utilisées.

**Ordre d'exécution important** : toutes les cellules doivent être exécutées dans l'ordre, de haut en bas,
avant la dernière cellule (`app.run(...)`) qui démarre le serveur et bloque le kernel.

Pour arrêter le serveur : interrompre le kernel. Pour relancer après une modification de code : *Run All*.

### 1. Import de Flask — le framework web

**Rôle** : Flask fournit le cœur du serveur : associer une URL à une fonction Python (le *routage*), construire les
réponses HTTP, et faire le lien avec les fichiers HTML via un moteur de templates.

**Origine et histoire** : Flask a été créé en 2010 par le développeur allemand Armin Ronacher, à l'origine comme un
prototype publié le 1er avril (un « poisson d'avril » technique). L'idée était de proposer une alternative légère à
Django (2005), qui impose beaucoup de structure (ORM, admin, authentification intégrés). Flask est un
*micro-framework* : il ne fournit que le strict nécessaire et laisse le développeur choisir ses propres outils.
Il s'appuie sur deux autres bibliothèques que Ronacher avait déjà écrites :
- **Werkzeug**, une boîte à outils WSGI (le protocole bas niveau entre un serveur web et une application Python)
- **Jinja2**, le moteur de templates qui permettra d'injecter des variables Python dans les fichiers `.html`

**Fonctionnement** : chaque nom importé a un rôle précis dans ce notebook :
- `Flask` : la classe application, instanciée une seule fois (`app = Flask(__name__)`, cellule 16)
- `Response` : construit une réponse HTTP « sur mesure » — indispensable pour le flux vidéo, qui n'est pas du HTML
- `render_template` : charge un fichier de `templates/` et y remplace les `{{ variable }}` par des valeurs Python
- `request` : représente la requête HTTP entrante (on l'utilisera pour lire le formulaire d'identification)
- `redirect`, `url_for` : gèrent la redirection HTTP (code 302) et génèrent une URL à partir du *nom* d'une route
  plutôt que de l'écrire en dur — si le chemin d'une route change un jour, les liens ne cassent pas

In [1]:
from flask import Flask, Response, render_template, request, redirect, url_for, session, jsonify

### 2. Imports de la chaîne de vision par ordinateur

**Rôle** : ces bibliothèques fournissent la capture/manipulation d'images (`cv2`), le calcul numérique vectoriel
(`numpy`), la recherche de fichiers (`glob`), et l'accès aux modèles de reconnaissance faciale (`insightface`).

**Origine et histoire** :
- **OpenCV** (`cv2`) a été lancé en 1999 par Gary Bradski chez Intel. Le but initial était presque publicitaire :
  démontrer des applications gourmandes en calcul pour stimuler la vente de processeurs Intel plus puissants.
  Le projet est devenu la bibliothèque de vision par ordinateur open source la plus utilisée au monde.
- **NumPy** a été créé en 2005 par Travis Oliphant, en fusionnant deux projets antérieurs concurrents
  (*Numeric*, 1995, et *Numarray*). Il introduit la structure `ndarray` (tableau multidimensionnel), fondation de
  quasiment tout l'écosystème scientifique Python (pandas, scikit-learn, PyTorch s'en inspirent ou l'utilisent).
- **InsightFace** est un projet de recherche open source (organisation *deepinsight*) né vers 2018, qui regroupe
  plusieurs travaux publiés par ses auteurs sur la détection et la reconnaissance faciale (ArcFace, RetinaFace,
  SCRFD — détaillés dans les cellules 6-7). `Face` est une simple structure de données (un conteneur) qui regroupe
  la boîte englobante (`bbox`), les points de repère du visage (`kps`, *keypoints* : yeux, nez, coins de la bouche),
  et plus tard l'embedding calculé. `model_zoo` est l'utilitaire qui télécharge et charge les poids pré-entraînés
  au format ONNX (*Open Neural Network Exchange*, format d'échange de modèles créé par Microsoft et Facebook en
  2017 pour rendre les modèles interopérables entre frameworks).

**Compléments ajoutés en cours de projet** (multi-caméras, tableau de bord) :
- **`threading`** (bibliothèque standard) — un thread dédié par caméra (`boucle_camera()`), plus un verrou
  global pour protéger l'accès concurrent aux modèles ONNX.
- **`time`** (bibliothèque standard) — pauses courtes (`time.sleep`) dans les boucles de capture/streaming.
- **`shutil`** (bibliothèque standard) — `shutil.disk_usage()`, pour l'indicateur de stockage du tableau de bord.
- **`psutil`** — bibliothèque tierce (pas standard, à installer via `pip`) donnant accès aux métriques système
  (CPU, mémoire) indépendamment du système d'exploitation.

**Outils et technologies utilisés** :
- **OpenCV**, **NumPy**, **glob**, **InsightFace**, **ONNX** (détaillés ci-dessus)
- **threading**, **time**, **shutil** (bibliothèque standard Python)
- **psutil** (bibliothèque tierce, métriques système)


In [2]:
import cv2
import threading
import time
import numpy as np
from glob import glob
from insightface.app.common import Face
from insightface.model_zoo import model_zoo
import shutil
import psutil
import io
from openpyxl import Workbook
from openpyxl.drawing.image import Image as ImageExcel
from openpyxl.styles import Font, PatternFill, Alignment
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, Image as ImagePdf
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

### 3. Imports MongoDB et utilitaires système

**Rôle** : `pymongo` est le pilote officiel permettant à Python de dialoguer avec MongoDB. `datetime` fournit les
horodatages des détections. `os` gère les chemins de fichiers et la création de dossiers. `json` sert à lire/écrire
les fichiers de configuration persistés sur disque (`data/cameras.json`, `data/admin.json`, `acces.json`).

**Origine** : `pymongo` est maintenu directement par MongoDB Inc. depuis la création de la base en 2009. `datetime`
et `os` font partie de la bibliothèque standard de Python depuis ses toutes premières versions (1991).

In [3]:
from pymongo import MongoClient
from datetime import datetime, timedelta
import os
import json
import re
from translation import textes
from flask import session

### 4. Configuration générale

**Rôle** : centraliser les constantes du projet — dossiers de sauvegarde des visages, seuil de décision, et la
correspondance entre le nom logique d'une caméra et sa source réelle.

**Principe mathématique — le seuil `SEUIL_DEFAUT`** : ce n'est pas une constante arbitraire, c'est un curseur qui
règle un compromis statistique classique en biométrie, entre deux types d'erreurs :
- le **FAR** (*False Acceptance Rate*) : accepter à tort un inconnu comme une personne connue (seuil trop bas)
- le **FRR** (*False Rejection Rate*) : rejeter à tort une personne connue comme inconnue (seuil trop haut)

En traçant FAR et FRR en fonction du seuil, leur point de croisement s'appelle l'**EER** (*Equal Error Rate*) —
une métrique standard pour évaluer un système biométrique. `0.5` est une valeur de départ raisonnable pour des
embeddings ArcFace normalisés, à ajuster empiriquement selon vos données réelles.

**Origine — DroidCam** : DroidCam est une application créée par la société Dev47Apps, qui transforme un smartphone
en webcam, exposée soit comme périphérique virtuel local (connexion USB, via un pilote/« client » installé sur le
PC), soit comme un flux HTTP accessible en réseau local (Wi-Fi, par défaut sur le port `4747`, au format MJPEG —
voir cellule 16 pour le détail de ce format).

**Mise à jour** : les caméras sont maintenant persistées dans `data/cameras.json`, pour survivre à un redémarrage du kernel (ajout/suppression via la page Paramètres).

In [4]:
DOSSIER_INCONNUS = "static/inconnus"  # sous static/ pour être servi par Flask
DOSSIER_SUCCES = "static/succes"
SEUIL_DEFAUT = 0.5
FICHIER_CAMERAS = "data/cameras.json"
# Logo affiché en en-tête des exports Excel/PDF (voir generer_reponse_excel / generer_reponse_pdf).
# Si ton fichier logo a un autre nom ou un autre emplacement, ajuste ce chemin en conséquence.
CHEMIN_LOGO = os.path.join("static", "icones images", "logo_inch.jpg")


def detecter_cameras_disponibles(nombre_max_a_tester=6):
    """
    Teste les index webcam de 0 à nombre_max_a_tester-1 (compatible Iriun Webcam,
    qui expose chaque téléphone comme un index de webcam classique, pas une URL).
    Retourne {"CAM-01": index, "CAM-02": index, ...} pour chaque index qui répond
    réellement (une frame a pu être lue), dans l'ordre où ils sont trouvés.
    """
    cameras_detectees = {}
    numero = 1
    for index in range(nombre_max_a_tester):
        cap = cv2.VideoCapture(index, cv2.CAP_MSMF)
        if cap.isOpened():
            ret, frame = cap.read()
            if ret and frame is not None:
                cameras_detectees[f"CAM-{numero:02d}"] = index
                numero += 1
        cap.release()
    return cameras_detectees


def charger_cameras():
    """
    Charge data/cameras.json s'il existe déjà (respecte les suppressions faites
    depuis Paramètres). Sinon, détecte automatiquement les caméras connectées —
    plus d'ajout manuel : le nombre d'espaces caméra dans l'interface correspond
    exactement au nombre de caméras physiquement détectées.
    """
    if os.path.exists(FICHIER_CAMERAS):
        with open(FICHIER_CAMERAS, "r", encoding="utf-8") as f:
            try:
                return json.load(f)
            except json.JSONDecodeError:
                print(f"{FICHIER_CAMERAS} est vide ou corrompu — nouvelle détection automatique.")

    print("Détection automatique des caméras connectées...")
    cameras_detectees = detecter_cameras_disponibles()
    sauvegarder_cameras(cameras_detectees)
    print(f"{len(cameras_detectees)} caméra(s) détectée(s) : {cameras_detectees}")
    return cameras_detectees


def sauvegarder_cameras(cameras):
    os.makedirs(os.path.dirname(FICHIER_CAMERAS), exist_ok=True)
    with open(FICHIER_CAMERAS, "w", encoding="utf-8") as f:
        json.dump(cameras, f, ensure_ascii=False, indent=2)


CAMERAS = charger_cameras()

os.makedirs(DOSSIER_INCONNUS, exist_ok=True)
os.makedirs(DOSSIER_SUCCES, exist_ok=True)

### 5. Connexion à MongoDB

**Rôle** : ouvrir la connexion vers le serveur MongoDB local et vérifier immédiatement qu'elle fonctionne.

**Origine et histoire** : MongoDB a été créé en 2007 par Dwight Merriman, Eliot Horowitz et Kevin Ryan, sous le nom
de société *10gen* (renommée MongoDB Inc. en 2013). Le nom « Mongo » vient de « humongous » (énorme) — l'objectif
initial était de gérer de très gros volumes de données avec un modèle plus flexible que les bases relationnelles
classiques (SQL). MongoDB appartient à la famille **NoSQL**, plus précisément aux bases *orientées documents* :
au lieu de lignes dans des tables à colonnes fixes, elle stocke des documents au format **BSON** (*Binary JSON*),
une extension binaire de JSON qui ajoute des types que JSON ne supporte pas nativement (dates, identifiants
binaires `ObjectId`, données binaires brutes). C'est ce qui permet ici de stocker un embedding (une liste de 512
nombres) directement comme un champ de document, sans schéma de table à définir à l'avance.

**Fonctionnement** : `MongoClient(...)` ouvre une connexion *paresseuse* (« lazy ») — elle ne vérifie rien tout de
suite. `client.admin.command("ping")` envoie une commande d'administration minimale (faisant partie du protocole
réseau natif de MongoDB, le *wire protocol*) pour forcer une vérification immédiate plutôt que de découvrir un
problème de connexion plus tard, au milieu d'une requête plus complexe.

In [5]:
client = MongoClient("mongodb://localhost:27017/")
db = client["surveillance"]

try:
    client.admin.command("ping")
    print("Connexion à MongoDB réussie.")
except Exception as e:
    print("Échec de connexion à MongoDB :", e)

Connexion à MongoDB réussie.


### 6. Chargement du modèle de détection (`det_10g.onnx` — SCRFD)

**Rôle** : ce modèle repère *où* se trouvent les visages dans une image — il ne les identifie pas, il les localise.
Pour chaque visage trouvé, il retourne une boîte englobante (`bbox`) et 5 points de repère (`kps` : les deux yeux,
le nez, les deux coins de la bouche).

**Origine et histoire** : `det_10g.onnx` implémente **SCRFD** (*Sample and Computation Redistribution for Efficient
Face Detection*), publié en 2021 par l'équipe InsightFace (Guo, Deng et al.). SCRFD s'inscrit dans une lignée de
détecteurs mono-étape (« single-stage ») remontant à **RetinaNet** (Facebook AI Research, 2017 — qui a introduit la
*focal loss* pour compenser le déséquilibre entre zones de fond et zones contenant un objet) et aux **FPN**
(*Feature Pyramid Networks*, 2016), qui permettent de détecter des visages à plusieurs échelles simultanément.
**RetinaFace** (Deng et al., 2019) a adapté cette architecture spécifiquement aux visages, en ajoutant la
régression des 5 points de repère en plus de la boîte. SCRFD est ensuite venu optimiser le rapport précision/vitesse
de RetinaFace en redistribuant intelligemment le calcul entre les différentes échelles du réseau.

**Principe mathématique** : comme la plupart des détecteurs modernes, le réseau ne prédit pas directement des
coordonnées absolues. Il définit une grille de points *ancres* sur l'image, et pour chacun prédit :
- un score de confiance (visage / pas visage), via une fonction sigmoïde
- un décalage `(dx, dy, dw, dh)` par rapport à une boîte de référence — principe de régression de boîte introduit
  par R-CNN et ses successeurs (2014-2015)

Comme un même visage peut être détecté par plusieurs ancres voisines, un post-traitement appelé **NMS**
(*Non-Maximum Suppression*) élimine les doublons : il garde la détection au score le plus élevé et supprime toute
autre boîte dont le recouvrement avec elle (mesuré par l'**IoU**, *Intersection over Union* — l'aire d'intersection
divisée par l'aire d'union des deux boîtes) dépasse un certain seuil. `max_num=0` dans nos appels signifie
« aucune limite sur le nombre de visages détectés ».

In [6]:
det_model = model_zoo.get_model("buffalo_l/det_10g.onnx", download=True)
det_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

c:\Users\NDOUNG CYNTHIA\AppData\Local\Programs\Python\Python310\lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


### 7. Chargement du modèle de reconnaissance (`w600k_r50.onnx` — ArcFace)

**Rôle** : contrairement au modèle précédent qui *localise*, celui-ci *caractérise* — il transforme un visage déjà
détecté et aligné en un vecteur numérique de 512 dimensions (l'**embedding**), une sorte d'empreinte mathématique
du visage. Deux photos de la même personne doivent produire deux vecteurs proches ; deux personnes différentes,
deux vecteurs éloignés.

**Origine et histoire** : le nom du fichier indique deux choses : `r50` = une architecture **ResNet-50**
(*Residual Network*, He et al., Microsoft Research, 2015 — l'article qui a introduit les *connexions résiduelles*,
permettant d'entraîner des réseaux beaucoup plus profonds en laissant le signal du gradient « sauter » certaines
couches pendant l'apprentissage, ce qui a résolu le problème du gradient qui s'évanouit dans les réseaux profonds).
`w600k` fait référence au jeu de données d'entraînement (dérivé de MS1M/Glint360k, plusieurs millions d'images
couvrant des centaines de milliers d'identités).

Le réseau est entraîné avec la fonction de perte **ArcFace** (*Additive Angular Margin Loss*, Deng, Guo, Xue,
Zafeiriou — 2019, InsightFace). C'est l'élément le plus important à comprendre mathématiquement :

**Principe mathématique — ArcFace** : un classifieur softmax classique compare un vecteur caractéristique à des
vecteurs de référence via leur produit scalaire, ce qui revient (une fois les vecteurs normalisés) à comparer des
**cosinus d'angles**. ArcFace ajoute une marge angulaire `m` directement à l'angle θ entre le vecteur et sa classe
correcte, avant de recalculer le cosinus : la perte optimise `cos(θ + m)` plutôt que `cos(θ)`. Concrètement, le
réseau est *forcé* pendant l'entraînement à rapprocher angulairement les visages d'une même personne beaucoup plus
qu'un entraînement softmax classique ne l'exigerait, et à repousser les personnes différentes plus loin sur la
sphère. Résultat : les embeddings d'une même identité se regroupent en un cône angulaire étroit, ce qui rend la
simple distance angulaire (donc la similarité cosinus, cellule 8) directement utilisable comme mesure de
reconnaissance — sans réseau de comparaison supplémentaire.

`face.normed_embedding` (utilisé plus loin) est déjà **normalisé** (norme euclidienne = 1) : tous les embeddings
vivent sur une sphère unité de dimension 512, ce qui est précisément l'hypothèse dont a besoin la similarité
cosinus pour être un simple produit scalaire (cellule 8).

In [7]:
rec_model = model_zoo.get_model("buffalo_l/w600k_r50.onnx", download=True)
rec_model.prepare(ctx_id=0, input_size=(640, 640), det_thres=0.5)

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}


### 8. `find_match()` — comparaison par similarité cosinus

**Rôle** : comparer l'embedding d'un visage détecté à tous les embeddings connus en base, et retourner le nom le
plus proche — ou `"Inconnu"` si même le plus proche reste trop loin.

**Principe mathématique — similarité cosinus** : pour deux vecteurs normalisés **a** et **b** (norme = 1), leur
produit scalaire `a · b = Σ aᵢbᵢ` est *exactement égal* à `cos(θ)`, où θ est l'angle entre les deux vecteurs. C'est
une conséquence directe de la définition géométrique du produit scalaire : `a · b = ‖a‖‖b‖cos(θ)`, qui se simplifie
ici puisque `‖a‖ = ‖b‖ = 1`. Une valeur de 1 signifie des vecteurs identiques en direction (angle nul), 0 signifie
des vecteurs orthogonaux (aucune corrélation), et -1 des vecteurs opposés. `np.dot(embedding, known_embeddings.T)`
calcule ce produit scalaire simultanément contre *tous* les embeddings connus (une multiplication matrice-vecteur),
ce qui est bien plus rapide qu'une boucle Python comparant un par un.

`np.argmax(scores)` sélectionne l'indice du score le plus élevé — c'est un classifieur du **plus proche voisin**
(*1-nearest neighbor*, ou *1-NN*), l'une des méthodes de classification les plus anciennes et les plus simples en
apprentissage automatique (formalisée par Cover et Hart en 1967 dans leur article fondateur sur la classification
par plus proche voisin). Le concept même de « similarité cosinus » pour comparer des vecteurs vient historiquement
du **modèle vectoriel** en recherche d'information (Gerard Salton, années 1970), où l'on représentait des documents
texte comme des vecteurs pour mesurer leur ressemblance — la même idée mathématique est réutilisée ici, appliquée
non pas à des mots mais aux embeddings faciaux.

Le `threshold` est le curseur FAR/FRR décrit en cellule 4 : sous ce seuil, même la meilleure correspondance n'est
pas jugée assez fiable et le visage est classé `"Inconnu"`.

In [8]:
def find_match(embedding, known_embeddings, known_names, threshold=SEUIL_DEFAUT):
    """Similarité cosinus entre un embedding test et la base connue.

    Si aucune base n'est encore chargée (known_embeddings est None, ou vide),
    retourne directement "Inconnu" au lieu de planter — évite que le flux
    vidéo ne s'interrompe simplement parce que db.Persons est vide.
    """
    if known_embeddings is None or len(known_embeddings) == 0:
        return "Inconnu", 0.0

    scores = np.dot(embedding, known_embeddings.T)
    scores = np.clip(scores, 0.0, 1.0)
    idx = np.argmax(scores)
    score = scores[idx]
    name = known_names[idx] if score > threshold else "Inconnu"
    return name, score

### 9. `agrandir_bbox_epaules()` — recadrage géométrique

**Rôle** : cette fonction n'a **aucun lien avec l'IA** — c'est de la géométrie pure. Elle agrandit la boîte du
visage détecté pour inclure les épaules, uniquement pour que l'image *sauvegardée sur disque* (dans `inconnus/`
ou `succes/`) soit plus lisible pour un humain qui la consulterait plus tard. L'embedding, lui, a déjà été calculé
à l'étape précédente sur le visage seul — cette fonction n'intervient jamais dans la reconnaissance elle-même.

**Fonctionnement mathématique** : chaque marge (`marge_haut`, `marge_bas`, `marge_cotes`) est un pourcentage de la
largeur ou hauteur de la boîte d'origine, ajouté de chaque côté, puis limité (`max(0, ...)` / `min(largeur_frame,
...)`) pour ne jamais sortir des limites de l'image. C'est une simple opération d'échelle et de translation,
sans aucun modèle statistique derrière — les valeurs par défaut (0.5, 0.8, 0.6) ont été choisies empiriquement pour
englober les épaules sans capturer trop d'arrière-plan.

In [9]:
def agrandir_bbox_epaules(bbox, frame_shape, marge=0.7):
    """
    Découpe un CARRÉ centré sur le CENTRE du visage (épaules incluses).

    Avant : les marges étaient asymétriques (haut 0.5, bas 0.8, côtés 0.6),
    donc l'image sauvegardée n'était ni carrée ni centrée sur le visage —
    une fois affichée dans une vignette carrée (object-fit: cover), le
    visage apparaissait décalé plutôt que centré.

    Maintenant : le centre du visage devient le centre exact du carré
    découpé, avec la même marge appliquée de tous les côtés. Le carré est
    dimensionné sur la plus grande dimension du visage détecté, pour rester
    cohérent même si les visages ont des proportions légèrement différentes.
    """
    x1, y1, x2, y2 = bbox.astype(int)
    h_frame, w_frame = frame_shape[:2]

    largeur = x2 - x1
    hauteur = y2 - y1
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2

    cote = int(max(largeur, hauteur) * (1 + 2 * marge))
    demi = cote // 2

    x1_e = max(0, cx - demi)
    y1_e = max(0, cy - demi)
    x2_e = min(w_frame, cx + demi)
    y2_e = min(h_frame, cy + demi)

    return x1_e, y1_e, x2_e, y2_e


### 10. `charger_embeddings_mongo()` — lecture de la base connue

**Rôle** : recharger en mémoire tous les embeddings enregistrés, sous la forme attendue par `find_match()` (une
matrice NumPy + une liste de noms parallèle).

**Fonctionnement** : `db.Persons.find({}, {"nom": 1, "embedding": 1})` est une requête MongoDB : le
premier argument `{}` signifie « aucun filtre, tous les documents », le second est une *projection* qui limite les
champs renvoyés (économie de bande passante — inutile de rapatrier `role`/`departement` ici). Le langage de requête
de MongoDB — des documents JSON décrivant le filtre — a été conçu délibérément pour ressembler à la syntaxe des
objets JavaScript, MongoDB ayant historiquement ciblé en priorité les développeurs web (Node.js) dans son adoption
initiale à la fin des années 2000.

`np.array([d["embedding"] for d in docs])` empile toutes les listes de 512 nombres en une seule matrice de forme
`(nombre_de_visages, 512)` — c'est cette matrice que `find_match()` utilise pour comparer un visage à tous les
autres en une seule opération matricielle plutôt qu'une boucle.

In [10]:
def charger_embeddings_mongo():
    """Recharge les embeddings connus depuis MongoDB. Retourne (None, None) si la base est vide."""
    docs = list(db.Persons.find({}, {"nom": 1, "embedding": 1}))
    if not docs:
        print("Aucun embedding trouvé dans MongoDB.")
        return None, None

    known_embeddings = np.array([d["embedding"] for d in docs])
    known_names = [d["nom"] for d in docs]
    return known_embeddings, known_names

### 11. `enregistrer_acces()` — journalisation d'une détection

**Rôle** : écrire une trace de chaque détection (connue ou inconnue) dans le journal d'accès, et — si la personne
n'est pas reconnue — créer en plus une fiche en attente d'identification humaine.

**Fonctionnement** : `insert_one(...)` est l'opération d'écriture la plus basique de MongoDB — elle correspond à
un `INSERT` en SQL, mais sans nécessiter de schéma de table prédéfini : chaque document peut en théorie avoir des
champs différents (ici on garde volontairement une structure cohérente pour simplifier les requêtes ultérieures).
C'est le principe même des bases *NoSQL orientées documents* : le schéma est appliqué par la logique applicative
(cette fonction Python), pas imposé rigidement par la base elle-même — flexibilité utile en développement rapide,
au prix d'une responsabilité accrue côté code pour garder les données cohérentes.

**Historique de la notion de « journal d'accès »** : le principe de consigner chronologiquement chaque événement
d'un système remonte au *logging* informatique classique — bien antérieur aux bases NoSQL — et reste le même ici :
chaque document représente un événement immuable, jamais modifié après coup, ce qui permet de reconstruire toute
l'historique d'activité (contrairement à une simple mise à jour de compteur qui perdrait le détail de chaque passage).

In [11]:
def enregistrer_acces(nom, statut, score, camera, image=None):
    """
    Ajoute une entrée dans le journal d'accès (MongoDB), avec la caméra source.
    Si le statut est REFUSE, ajoute aussi une fiche dans personnes_inconnues
    en attente d'identification.
    """
    db.Detections.insert_one({
        "date": datetime.now().strftime("%Y-%m-%d"),
        "heure": datetime.now().strftime("%H:%M:%S"),
        "nom": nom,
        "statut": statut,
        "score_detection": round(float(score), 4),
        "image": image,
        "camera": camera,
    })

    if statut == "REFUSE":
        db.UnknownPersons.insert_one({
            "score": round(float(score), 4),
            "image": image,
            "date": datetime.now(),
            "camera": camera,
            "traite": False,
        })

### 12. `charger_stats_du_jour()` — agrégation pour le tableau de bord

**Rôle** : calculer, à la demande, les chiffres du jour (total, connus, inconnus, première/dernière détection) sans
jamais les stocker à l'avance — ils sont recalculés depuis le journal brut chaque fois que la page dashboard est
visitée.

**Principe** : c'est une **agrégation** au sens des bases de données — transformer un ensemble d'enregistrements
détaillés en quelques statistiques résumées. Ici l'agrégation est faite « à la main » en Python après avoir
rapatrié les documents (`sum(1 for l in ... if ...)` est un simple comptage conditionnel), plutôt qu'avec le
framework d'agrégation natif de MongoDB (`$group`, `$match`, etc., un pipeline inspiré des pipelines Unix). Pour un
volume de données de l'ordre de quelques centaines à quelques milliers de documents par jour, les deux approches
sont équivalentes en pratique ; le pipeline natif Mongo devient préférable si le volume grossit beaucoup, car il
évite de transférer tous les documents bruts vers Python avant de les résumer.

In [12]:
def charger_stats_du_jour():
    """Statistiques du jour pour le dashboard/temps réel : total, connus, inconnus, dernières détections."""
    aujourdhui = datetime.now().strftime("%Y-%m-%d")
    logs_du_jour = list(db.Detections.find({"date": aujourdhui}).sort("heure", 1))

    total = len(logs_du_jour)
    noms_connus_distincts = {l["nom"] for l in logs_du_jour if l["statut"] == "SUCCES"}
    connus = len(noms_connus_distincts)  # personnes uniques, pas le nombre d'événements
    inconnus = total - sum(1 for l in logs_du_jour if l["statut"] == "SUCCES")

    premiere = logs_du_jour[0]["heure"] if logs_du_jour else None
    derniere = logs_du_jour[-1]["heure"] if logs_du_jour else None

    dernieres = logs_du_jour[-10:][::-1]  # les 10 plus récentes, ordre décroissant
    for l in dernieres:
        l["id"] = str(l["_id"])  # utilisé comme ancre HTML pour la vignette plein écran

    return {
        "total": total,
        "connus": connus,
        "inconnus": inconnus,
        "premiere_detection": premiere,
        "derniere_detection": derniere,
        "dernieres": dernieres,
    }


def charger_toutes_detections_du_jour():
    """
    Toutes les détections d'aujourd'hui, de la plus récente à la plus
    ancienne — contrairement à charger_stats_du_jour() (limitée aux 10
    dernières pour l'aperçu temps réel/dashboard), utilisée par la page
    Détections qui doit afficher la liste complète.
    """
    aujourdhui = datetime.now().strftime("%Y-%m-%d")
    logs = list(db.Detections.find({"date": aujourdhui}).sort("heure", -1))
    for l in logs:
        l["id"] = str(l["_id"])
    return logs


### 13. `charger_inconnus_non_traites()` — file d'attente d'identification

**Rôle** : récupérer uniquement les fiches d'inconnus pas encore résolues, pour la page `unknowns.html`.

**Fonctionnement** : `{"traite": False}` est un filtre d'égalité — le plus simple des opérateurs de requête
MongoDB. `.sort("date", -1)` trie par ordre décroissant (les plus récentes en premier ; `1` donnerait l'ordre
croissant). C'est l'équivalent conceptuel d'une clause `WHERE traite = false ORDER BY date DESC` en SQL — la
syntaxe diffère (documents vs. clauses textuelles) mais l'opération logique est identique.

### 12.1 `calculer_stats_systeme()`

**Rôle** : calculer les 3 indicateurs affichés en haut de la vue temps réel (admin et employé) — état du
système, détections par minute, latence du flux — à partir de données réelles plutôt que de valeurs fixes.

**Fonctionnement** :
- **Détections/min** : nombre de documents dans `Detections` avec une heure dans la dernière minute.
- **Latence** : temps écoulé depuis la dernière frame reçue par `boucle_camera()`, moyenné sur les caméras
  actives (`dernieres_maj_frames`).
- **État** : "Opérationnel" si la latence moyenne est inférieure à 2 secondes, "Dégradé" au-delà, "Hors ligne"
  si aucune caméra n'a encore renvoyé de frame.

In [13]:
def calculer_stats_systeme():
    maintenant = datetime.now()

    aujourdhui = maintenant.strftime("%Y-%m-%d")
    il_y_a_1_min = (maintenant - timedelta(minutes=1)).strftime("%H:%M:%S")
    detections_par_min = db.Detections.count_documents({
        "date": aujourdhui,
        "heure": {"$gte": il_y_a_1_min},
    })

    with verrou_frames:
        derniers = dict(dernieres_maj_frames)

    if derniers:
        latences_ms = [(maintenant - t).total_seconds() * 1000 for t in derniers.values()]
        latence_ms = int(sum(latences_ms) / len(latences_ms))
    else:
        latence_ms = None

    if latence_ms is None:
        etat = "Hors ligne"
    elif latence_ms < 2000:
        etat = "Opérationnel"
    else:
        etat = "Dégradé"

    return {
        "etat": etat,
        "detections_par_min": detections_par_min,
        "latence_ms": latence_ms,
    }

In [14]:
def charger_inconnus_non_traites():
    """Liste des détections inconnues en attente d'identification, pour la page unknowns."""
    return list(db.UnknownPersons.find({"traite": False}).sort("date", -1))

### 13.1 Prétraitement du contraste — CLAHE

**Rôle** : égaliser le contraste de chaque image avant détection/reconnaissance, pour stabiliser les résultats
sous des éclairages très inégaux (webcam en contre-jour, DroidCam/Iriun avec une caméra de téléphone moins
maîtrisée qu'une caméra dédiée).

**Fonctionnement** : CLAHE (*Contrast Limited Adaptive Histogram Equalization*) égalise l'histogramme des
intensités **localement**, par petites tuiles de l'image (ici 8×8), plutôt que sur l'image entière — une
égalisation globale accentuerait le bruit dans les zones déjà bien exposées. Le "Contrast Limited" plafonne
l'amplification du contraste par tuile, pour éviter de sur-amplifier le bruit dans les zones très sombres ou
très claires. Appliqué uniquement sur le canal de **luminance (L)** de l'espace colorimétrique **LAB** — pas
directement sur les canaux Rouge/Vert/Bleu — pour ne pas fausser les couleurs de peau, un indice implicitement
utilisé par le réseau de reconnaissance.

**Origine et histoire** : l'égalisation d'histogramme adaptative a été formalisée dans les années 1980
(Pizer et al., 1987), pour l'imagerie médicale où le contraste local est critique (radiographies). La variante
"Contrast Limited" a été ajoutée peu après pour corriger le sur-bruitage observé sur les images à faible
variation locale. C'est aujourd'hui une fonction standard d'OpenCV (`cv2.createCLAHE`).

**Où c'est appliqué dans ce projet** : sur chaque photo du Dataset au moment de l'extraction des embeddings
(cellule 14), et sur chaque frame de caméra en direct (`boucle_camera()`) — pour que la reconnaissance en
temps réel "voie" le même type d'image que celle utilisée pour construire la base de référence.

**Outils et technologies utilisés** :
- **OpenCV** (`cv2.createCLAHE`, `cv2.cvtColor`, `cv2.split`/`cv2.merge`) — égalisation adaptative de contraste
- **NumPy** (implicite, les images OpenCV sont des tableaux NumPy)

In [15]:
def appliquer_clahe(image_bgr):
    """
    Égalise le contraste de l'image avec CLAHE, sur le canal de luminance (L)
    de l'espace LAB uniquement — les couleurs (canaux a/b) ne sont pas touchées.
    """
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_egalise = clahe.apply(l)

    lab_egalise = cv2.merge((l_egalise, a, b))
    return cv2.cvtColor(lab_egalise, cv2.COLOR_LAB2BGR)

### 13.2 Data Augmentation du Dataset

**Rôle** : générer plusieurs variantes de chaque photo du Dataset (miroir, luminosité, légère rotation) avant
extraction des embeddings, pour que la base de référence par personne couvre davantage de conditions
(éclairage, angle) qu'une seule photo ne peut en capturer — sans avoir à prendre physiquement plus de photos.

**Fonctionnement** : pour une photo source, 5 variantes supplémentaires sont produites : un miroir horizontal,
deux versions de luminosité (plus sombre / plus claire, via `cv2.convertScaleAbs`), et deux légères rotations
(±10°, via une matrice de rotation `cv2.getRotationMatrix2D` + `cv2.warpAffine`). Chaque variante (CLAHE déjà
appliqué) produit son **propre embedding**, ajouté à la base de référence de la personne — une seule photo
source peut ainsi générer jusqu'à 6 embeddings au lieu d'un seul.

**Origine et histoire** : la data augmentation est une pratique standard en apprentissage automatique depuis
les débuts des réseaux de neurones convolutifs (popularisée notamment par AlexNet, Krizhevsky et al., 2012),
pour compenser un jeu de données limité en générant artificiellement de la variabilité, sans risquer le
surapprentissage (*overfitting*) qu'une simple duplication de données identiques provoquerait.

**Attention** : ici, on n'entraîne pas un modèle — `det_model`/`rec_model` sont déjà pré-entraînés (ArcFace).
L'augmentation sert uniquement à enrichir la **galerie de référence** (les embeddings connus stockés dans
`Persons`), pas à ré-entraîner le réseau lui-même.

**Outils et technologies utilisés** :
- **OpenCV** — `cv2.flip`, `cv2.convertScaleAbs`, `cv2.getRotationMatrix2D`, `cv2.warpAffine`
- **NumPy** (implicite, via les tableaux OpenCV)

In [16]:
def augmenter_image(image_bgr):
    """
    Génère l'image originale + 5 variantes (miroir, luminosité, rotation),
    pour enrichir la base d'embeddings connus à partir d'une seule photo.
    """
    variantes = [image_bgr]

    # Miroir horizontal
    variantes.append(cv2.flip(image_bgr, 1))

    # Luminosité : plus sombre, puis plus claire
    for facteur in (0.8, 1.2):
        variantes.append(cv2.convertScaleAbs(image_bgr, alpha=facteur, beta=0))

    # Légère rotation, dans les deux sens
    hauteur, largeur = image_bgr.shape[:2]
    centre = (largeur // 2, hauteur // 2)
    for angle in (-10, 10):
        matrice_rotation = cv2.getRotationMatrix2D(centre, angle, 1.0)
        variante = cv2.warpAffine(
            image_bgr, matrice_rotation, (largeur, hauteur), borderMode=cv2.BORDER_REPLICATE
        )
        variantes.append(variante)

    return variantes

### 14. `extract_face_embeddings()` — enrôlement par lot

**Rôle** : traiter un dossier `Dataset/<Personne>/*.jpg` pour générer les embeddings de référence de chaque
personne connue, et les insérer en base. C'est l'équivalent de la phase d'**enrôlement** (*enrollment*) en
biométrie — le moment où un système apprend à qui appartient quel visage, avant toute reconnaissance ultérieure.
Ce concept d'enrôlement précède largement le deep learning : il structure historiquement tous les systèmes
biométriques, y compris les plus anciens systèmes d'empreintes digitales (AFIS, *Automated Fingerprint
Identification Systems*, dès les années 1960-1970).

**Fonctionnement** : pour chaque personne (un sous-dossier), la fonction relit chaque photo, détecte le premier
visage trouvé (`bboxes[0]` — hypothèse simplificatrice qu'une photo d'enrôlement ne contient qu'un seul visage
pertinent), calcule son embedding, puis remplace en base tous les anciens embeddings de cette personne
(`delete_many` puis `insert_many`) par les nouveaux — ce mécanisme anti-doublon garantit qu'une ré-exécution sur
les mêmes photos ne fait pas grossir indéfiniment la base.

In [17]:
def extract_face_embeddings(dataset_dir="Dataset"):
    """
    Parcourt Dataset/<Personne>/*.jpg, extrait un embedding par image
    et remplace les embeddings existants de cette personne dans MongoDB
    (collection personnes_connues) pour éviter les doublons.
    """
    person_dirs = [
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ]
    print("Extraction des embeddings...")

    for person_name in person_dirs:
        directory = os.path.join(dataset_dir, person_name)
        img_paths = glob(f"{directory}/*.jpg")
        nouveaux_documents = []

        for img_path in img_paths:
            img = cv2.imread(img_path)
            if img is None:
                print(f"  Impossible de lire {img_path}")
                continue

            # Data Augmentation : 1 photo source -> jusqu'à 6 variantes (miroir,
            # luminosité, rotation), chacune passée au CLAHE avant extraction —
            # enrichit la base de référence sans avoir à reprendre plus de photos.
            for variante in augmenter_image(img):
                variante = appliquer_clahe(variante)

                bboxes, kpss = det_model.detect(variante, max_num=0, metric="default")
                if len(bboxes) == 0:
                    continue  # variante sans visage détectable (ex. rotation trop marquée) : ignorée silencieusement

                bbox = bboxes[0, :4]
                det_score = bboxes[0, 4]
                kps = kpss[0]
                face = Face(bbox=bbox, kps=kps, det_score=det_score)
                rec_model.get(variante, face)

                if hasattr(face, "normed_embedding"):
                    nouveaux_documents.append({
                        "nom": person_name,
                        "role": "",
                        "departement": "",
                        "embedding": face.normed_embedding.tolist(),
                    })

        if nouveaux_documents:
            supprimes = db.Persons.delete_many({"nom": person_name})
            db.Persons.insert_many(nouveaux_documents)
            print(f"  {person_name} : {supprimes.deleted_count} ancien(s) embedding(s) remplacé(s) par {len(nouveaux_documents)} nouveau(x) (photos sources + variantes augmentées).")

    print("Tous les embeddings ont été extraits et sauvegardés dans MongoDB.")

    # Recharge immédiatement la base en mémoire utilisée par la reconnaissance
    # en direct (boucle_camera()) — sinon les nouveaux visages resteraient
    # "Inconnu" jusqu'au prochain redémarrage du kernel.
    global known_embeddings, known_names
    known_embeddings, known_names = charger_embeddings_mongo()

In [18]:
# Décommenter pour (re)générer la base à partir du dossier Dataset/
#extract_face_embeddings()

### 15. Création de l'application Flask et principe du routage

**Rôle** : instancier l'objet central de Flask, qui va ensuite recevoir toutes les routes définies dans les
cellules suivantes.

**Principe — le décorateur `@app.route(...)`** : chaque route Flask s'appuie sur les **décorateurs**, une
fonctionnalité du langage Python normalisée par la *PEP 318* en 2003. Un décorateur est une fonction qui
« enveloppe » une autre fonction pour lui ajouter un comportement sans modifier son code — ici, `@app.route("/x")`
enregistre la fonction qui suit dans une table interne à Flask associant l'URL `/x` à cette fonction. Quand une
requête HTTP arrive sur `/x`, Flask consulte cette table et appelle la bonne fonction : c'est le principe du
**routage** URL, central dans tous les frameworks web modernes (popularisé notamment par Ruby on Rails en 2004,
puis largement repris, y compris par Django et Flask).

In [19]:
app = Flask(__name__)

# Nécessaire pour signer les cookies de session (session["role"], etc.).
# En production, cette clé devrait venir d'une variable d'environnement,
# jamais être codée en dur dans le notebook.
app.secret_key = "change-moi-en-production"

In [20]:
# route pour la traduction en francais et en anglais
@app.route("/choix_langue/<lang>",methods=["GET"])
def choix_langue(lang):
    url_actuel = request.referrer or url_for("realtime")
    if lang not in ("fr","en"):
        return redirect(url_actuel) 
    else: 
        session["lang"]=lang
        return redirect(url_actuel)
    
    
#fonction de traduction
def translate(page,cle,**kwargs):
    langue = session.get("lang", "fr")
    dict_langue = textes.get(langue, {})       # va chercher le bon dictionnaire de langue
    dict_page = dict_langue.get(page,{})     # le même principe avec dict_langue
    resultat = dict_page.get(cle,f"[{cle}]")        # va chercher "cle" dans dict_page, avec un repli f"[{cle}]"
    
    return resultat.format(**kwargs)

@app.context_processor
def injecter_traduction():
    return {"translate":translate}


### 16. `generer_frames()` — le flux vidéo MJPEG

**Rôle** : capturer les images de la caméra choisie en continu, appliquer la détection + reconnaissance sur chaque
image, dessiner les rectangles de couleur, journaliser chaque détection, puis transmettre l'image résultante au
navigateur — image par image, indéfiniment.

**Origine et histoire — le format MJPEG en streaming HTTP** : la technique utilisée ici (`multipart/x-mixed-
replace`) a été introduite par Netscape Communications en **1995**, à l'origine pour créer des animations « push »
dans le navigateur Netscape Navigator (le serveur poussait une nouvelle image, qui remplaçait la précédente, sans
que la page ait besoin de se recharger — un ancêtre direct des animations et du contenu dynamique côté serveur).
Cette même technique a ensuite été largement réutilisée par les premières caméras IP et logiciels de webcam dans
les années 2000, précisément parce qu'elle ne nécessite **aucun codec vidéo** : chaque image est indépendamment une
image JPEG classique, envoyée à la suite des autres, séparée par une balise de délimitation (*boundary*). C'est
beaucoup plus simple à mettre en œuvre qu'un vrai flux vidéo compressé (comme le H.264 utilisé par WebRTC, un
protocole bien plus récent, développé par Google et normalisé par le W3C à partir de 2011), au prix d'une
consommation de bande passante beaucoup plus élevée puisqu'aucune compression n'est faite *entre* les images
(pas de compensation de mouvement comme dans un codec vidéo).

**Fonctionnement du protocole** : la structure `b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes +
b"\r\n"` respecte la norme MIME multipart (RFC 2046) : une ligne de délimitation (`--frame`), un en-tête
décrivant le type de contenu qui suit, une ligne vide obligatoire, puis les octets bruts de l'image, et on
recommence pour l'image suivante. Le navigateur, en recevant un en-tête HTTP `Content-Type: multipart/x-mixed-
replace; boundary=frame` (cellule 18), sait qu'il doit afficher chaque partie à la place de la précédente au fur
et à mesure qu'elle arrive.

**Le mot-clé `yield`** : cette fonction est un **générateur** Python (introduit par la *PEP 255* en 2001). Au lieu
de retourner toutes les images capturées d'un coup (ce qui serait impossible, le flux est infini), elle *suspend*
son exécution à chaque `yield` et la reprend exactement là où elle s'était arrêtée à l'appel suivant — c'est ce
mécanisme qui permet à Flask de renvoyer un flux continu sans jamais charger toute la vidéo en mémoire.

### 15.1 `doit_capturer()` — limite les captures à une toutes les 10 secondes

**Rôle** : avant, chaque frame où un visage restait dans le champ (jusqu'à ~20x/sec) déclenchait
une nouvelle capture (image sauvegardée + entrée journalisée) — cette fonction limite désormais
les captures à une toutes les 10 secondes, par caméra et par identité reconnue.

**Limite connue** : toutes les personnes non reconnues partagent la même clé `"Inconnu"` par
caméra (impossible de les distinguer avant identification) — si deux inconnus différents passent
devant la même caméra à moins de 10 sec d'écart, seul le premier est capturé.


In [21]:
INTERVALLE_CAPTURE = timedelta(seconds=10)
dernieres_captures = {}  # (nom_camera, nom_ou_"Inconnu") -> datetime de la dernière capture
verrou_captures = threading.Lock()


def doit_capturer(nom_camera, pred_name):
    """
    Autorise une nouvelle capture seulement si 10 secondes se sont écoulées
    depuis la dernière capture pour cette caméra + cette identité.
    """
    cle = (nom_camera, pred_name)
    maintenant = datetime.now()
    with verrou_captures:
        derniere = dernieres_captures.get(cle)
        if derniere is not None and (maintenant - derniere) < INTERVALLE_CAPTURE:
            return False
        dernieres_captures[cle] = maintenant
        return True


### 15.2 `qualite_visage_suffisante()` — ne garder que les visages de face

**Rôle** : rejeter les visages détectés de profil, la tête penchée ou fortement baissée/relevée — un visage
techniquement "détecté" (score de confiance correct) mais mal orienté nuit à la qualité de la reconnaissance
et n'a pas d'intérêt à être capturé/journalisé.

**Fonctionnement** : à partir des 5 points de repère déjà fournis par SCRFD (yeux gauche/droit, nez, coins de
la bouche gauche/droit), trois critères géométriques simples sont vérifiés :
- **Yaw** (tête tournée à gauche/droite) : la distance horizontale nez↔œil doit être à peu près symétrique
  des deux côtés — un fort déséquilibre trahit un profil.
- **Roll** (tête penchée sur le côté) : l'angle de la ligne reliant les deux yeux ne doit pas dépasser un
  seuil (20° par défaut).
- **Pitch** (tête baissée/relevée) : le rapport entre la distance yeux→nez et nez→bouche doit rester dans
  une plage raisonnable — une tête baissée compresse fortement l'une des deux distances.

Ces trois vérifications sont des heuristiques géométriques simples (pas un modèle de pose entraîné) — rapides
à calculer, suffisantes pour filtrer les cas les plus nets, mais moins précises qu'un vrai estimateur de pose
3D sur les cas ambigus.

**Où c'est appliqué** : dans `boucle_camera()`, juste après la détection et avant la reconnaissance — un
visage jugé insuffisamment de face est ignoré (aucune reconnaissance, aucune capture, aucun cadre dessiné).

**Outils et technologies utilisés** :
- **NumPy** — `np.arctan2`, `np.degrees` pour l'angle de roll
- Géométrie 2D simple (aucune bibliothèque de pose faciale dédiée)

In [22]:
def qualite_visage_suffisante(kps, seuil_symetrie_yaw=0.5, angle_roll_max=20, ratio_pitch_min=0.4, ratio_pitch_max=2.5):
    """
    Détermine si un visage est suffisamment de face pour être capturé, à partir
    des 5 points de repère (œil gauche, œil droit, nez, bouche gauche, bouche
    droite) renvoyés par det_model.detect(). Rejette profils marqués, têtes
    penchées et têtes fortement baissées/relevées.
    """
    oeil_gauche, oeil_droit, nez, bouche_gauche, bouche_droite = kps

    # Yaw : symétrie des distances horizontales nez -> œil de chaque côté
    dist_gauche = abs(nez[0] - oeil_gauche[0])
    dist_droite = abs(oeil_droit[0] - nez[0])
    plus_petit, plus_grand = sorted([dist_gauche, dist_droite])
    if plus_grand == 0 or (plus_petit / plus_grand) < seuil_symetrie_yaw:
        return False  # visage de profil

    # Roll : angle de la ligne des yeux par rapport à l'horizontale
    angle_roll = np.degrees(np.arctan2(oeil_droit[1] - oeil_gauche[1], oeil_droit[0] - oeil_gauche[0]))
    if abs(angle_roll) > angle_roll_max:
        return False  # tête penchée sur le côté

    # Pitch : équilibre vertical entre yeux -> nez et nez -> bouche
    y_yeux = (oeil_gauche[1] + oeil_droit[1]) / 2
    y_bouche = (bouche_gauche[1] + bouche_droite[1]) / 2
    distance_haute = nez[1] - y_yeux
    distance_basse = y_bouche - nez[1]
    if distance_haute <= 0 or distance_basse <= 0:
        return False  # géométrie incohérente (visage très fortement incliné)

    ratio_pitch = distance_haute / distance_basse
    if ratio_pitch < ratio_pitch_min or ratio_pitch > ratio_pitch_max:
        return False  # tête baissée ou relevée

    return True

In [23]:
# Verrou global : les sessions ONNX Runtime de det_model/rec_model ne sont pas
# garanties thread-safe. Sans ce verrou, plusieurs caméras détectées en
# parallèle (plusieurs threads) peuvent faire échouer silencieusement la
# détection — aucun visage trouvé, donc aucun cadre dessiné, sans erreur.
verrou_modele = threading.Lock()

# Dernière frame encodée (JPEG) disponible pour chaque caméra, mise à jour en
# continu par boucle_camera() — indépendamment du fait que quelqu'un regarde
# la page temps réel ou non.
frames_actuelles = {}
dernieres_maj_frames = {}  # nom_camera -> datetime de la dernière frame reçue
verrou_frames = threading.Lock()


def boucle_camera(nom_camera, source):
    """
    Tourne indéfiniment dans un thread dédié à CETTE caméra : capture, détecte,
    annote, et garde toujours la dernière image prête à être servie. Un visage
    détecté mais insuffisamment de face (profil, tête penchée/baissée — voir
    qualite_visage_suffisante()) est entièrement ignoré : ni reconnu, ni
    capturé, ni dessiné. Pour les visages retenus, le rectangle + le nom sont
    dessinés à CHAQUE frame (fluidité du flux en direct), mais l'écriture sur
    disque + le journal ne se déclenchent qu'au plus une fois toutes les 10
    secondes par identité (voir doit_capturer() ci-dessus).
    """
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Impossible d'ouvrir la caméra {nom_camera}.")
        return

    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            time.sleep(1)  # la caméra peut se reconnecter ; on retente sans boucler à vide
            continue

        # Même prétraitement de contraste que sur les photos du Dataset (cellule 13.1) :
        # la reconnaissance en direct doit "voir" le même type d'image que celle qui a
        # servi à construire la base d'embeddings, sinon CLAHE côté Dataset seul n'aide pas.
        frame = appliquer_clahe(frame)

        with verrou_modele:
            bboxes, kpss = det_model.detect(frame, max_num=0, metric="default")

            for i in range(len(bboxes)):
                bbox = bboxes[i, :4]
                kps = kpss[i]

                if not qualite_visage_suffisante(kps):
                    continue  # profil, tête penchée ou baissée : ni reconnu, ni capturé, ni dessiné

                face = Face(bbox=bbox, kps=kps, det_score=bboxes[i, 4])
                rec_model.get(frame, face)
                test_embedding = face.normed_embedding

                pred_name, match_score = find_match(
                    test_embedding, known_embeddings, known_names, SEUIL_DEFAUT
                )

                if doit_capturer(nom_camera, pred_name):
                    x1_e, y1_e, x2_e, y2_e = agrandir_bbox_epaules(bbox, frame.shape)
                    visage = frame[y1_e:y2_e, x1_e:x2_e].copy()
                    horodatage = datetime.now().strftime("%Y%m%d_%H%M%S")

                    if pred_name == "Inconnu":
                        nom_fichier = f"inconnu_{horodatage}.jpg"
                        cv2.imwrite(os.path.join(DOSSIER_INCONNUS, nom_fichier), visage)
                        enregistrer_acces("Inconnu", "REFUSE", match_score, camera=nom_camera, image=nom_fichier)
                    else:
                        nom_fichier = f"{pred_name}_{horodatage}.jpg"
                        cv2.imwrite(os.path.join(DOSSIER_SUCCES, nom_fichier), visage)
                        enregistrer_acces(pred_name, "SUCCES", match_score, camera=nom_camera, image=nom_fichier)

                if pred_name == "Inconnu":
                    color = (0, 0, 255)
                    label = pred_name
                else:
                    color = (0, 255, 0)
                    label = f"{pred_name} ({match_score:.2f})"

                x1, y1, x2, y2 = map(int, bbox)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_DUPLEX, 0.6, color, 2)

        ret, buffer = cv2.imencode(".jpg", frame)
        if ret:
            with verrou_frames:
                frames_actuelles[nom_camera] = buffer.tobytes()
                dernieres_maj_frames[nom_camera] = datetime.now()


def demarrer_cameras():
    """Lance un thread boucle_camera() par caméra définie dans CAMERAS.
    À appeler une seule fois, juste avant app.run()."""
    for nom_camera, source in CAMERAS.items():
        t = threading.Thread(target=boucle_camera, args=(nom_camera, source), daemon=True)
        t.start()
        print(f"Thread de détection démarré pour {nom_camera}.")


def generer_frames(nom_camera):
    """
    Sert au navigateur la dernière frame déjà annotée par boucle_camera() —
    ne capture plus rien elle-même. Résultat : plusieurs onglets/personnes
    peuvent regarder la même caméra sans multiplier les connexions physiques,
    et la détection continue même si personne ne regarde.
    """
    while True:
        with verrou_frames:
            frame_bytes = frames_actuelles.get(nom_camera)

        if frame_bytes is not None:
            yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes + b"\r\n")

        time.sleep(0.05)  # ~20 images/seconde ; ajuste si besoin


### 17. Route `/video_feed/<nom_camera>` — exposer le flux au navigateur

**Rôle** : brancher le générateur précédent sur une URL HTTP consultable par une balise `<img>` dans les pages
`realtime.html` et `vue_users.html`.

**Fonctionnement** : `<nom_camera>` dans le chemin de la route est une **variable d'URL** — Flask capture
automatiquement tout ce qui apparaît à cet endroit et le passe comme argument à la fonction (`CAM-01`,
`CAM-02`, etc.). `mimetype="multipart/x-mixed-replace; boundary=frame"` est l'en-tête HTTP `Content-Type` qui
indique au navigateur le format spécial décrit dans la cellule précédente — sans cet en-tête précis, le navigateur
tenterait d'afficher le flux comme une image fixe unique et échouerait.

In [24]:
@app.route("/video_feed/<nom_camera>")
def video_feed(nom_camera):
    # Pas de redirect ici : une balise <img src="..."> ne peut pas suivre une
    # redirection vers une page HTML. On renvoie simplement un refus d'accès.
    if session.get("role") not in ("admin", "employe"):
        return Response(status=403)
    return Response(generer_frames(nom_camera), mimetype="multipart/x-mixed-replace; boundary=frame")

### 18. Route `/` — page de connexion

**Rôle** : servir la page de connexion statique `login.html`.

**Fonctionnement** : `render_template("login.html")` cherche le fichier dans le dossier `templates/` (convention
imposée par Flask) et renvoie son contenu tel quel — aucune variable n'est injectée ici, contrairement aux routes
suivantes.

In [25]:
@app.route("/")
def login():
    return render_template("login.html")

### 18.1 Décorateurs `require_admin` / `require_employe`

**Rôle** : protéger une route en vérifiant `session["role"]` avant d'exécuter la vue. Si la session ne correspond
pas au rôle attendu, l'utilisateur est renvoyé vers la page de connexion au lieu d'accéder à la page.

In [26]:
from functools import wraps


def require_admin(vue):
    @wraps(vue)
    def wrapper(*args, **kwargs):
        if session.get("role") != "admin":
            return redirect(url_for("login"))
        return vue(*args, **kwargs)
    return wrapper


def require_employe(vue):
    @wraps(vue)
    def wrapper(*args, **kwargs):
        if session.get("role") != "employe":
            return redirect(url_for("login"))
        return vue(*args, **kwargs)
    return wrapper

### 18.2 Routes `/login/admin` et `/login/employe`

**Rôle** : traiter la soumission des deux formulaires de `login.html`. L'admin est vérifié par le mot de passe
commun uniquement (`MOT_DE_PASSE_ADMIN`) — n'importe qui connaissant ce mot de passe peut se connecter en tant
qu'admin, le nom saisi ne sert qu'à l'afficher dans l'interface, pas à filtrer l'accès. L'employé, lui, est
vérifié par son nom, qui doit exister dans `db.Persons`. En cas de succès, la session est marquée et
l'utilisateur est redirigé vers son tableau de bord respectif.

**Outils et technologies utilisés** :
- **Flask** — `request.form`, `session`, `redirect`, `url_for`
- **Werkzeug** — `check_password_hash`
- **PyMongo** — `db.Persons.find_one()` (uniquement pour l'employé)


In [27]:
@app.route("/login/admin", methods=["POST"])
def login_admin():
    """
    Accès admin ouvert à tout le monde : le nom sert juste à identifier qui est
    connecté (affiché dans la sidebar), pas à filtrer qui a le droit d'entrer —
    seul le mot de passe commun (MOT_DE_PASSE_ADMIN) fait office de barrière.
    """
    nom = request.form.get("nom", "").strip()
    password = request.form.get("password", "")

    if not nom or not check_password_hash(MOT_DE_PASSE_ADMIN["password_hash"], password):
        return render_template("login.html", erreur_admin="Nom ou mot de passe incorrect.")

    session["role"] = "admin"
    session["nom"] = nom
    return redirect(url_for("realtime"))


@app.route("/login/employe", methods=["POST"])
def login_employe():
    nom = request.form.get("name", "").strip()

    personne = db.Persons.find_one({"nom": nom})

    if personne is None:
        return render_template("login.html", erreur_employe="Nom introuvable parmi les personnes connues.")

    session["role"] = "employe"
    session["nom"] = nom
    return redirect(url_for("vue_users"))

### 18.3 Route `/logout`

**Rôle** : vider la session (déconnexion), pour les deux rôles.

In [28]:
@app.route("/logout")
def logout():
    session.clear()
    return redirect(url_for("login"))

### 19. Route `/dashboard`

**Rôle** : assembler les données de **quatre fonctions différentes** pour alimenter `dashboard.html` — pas une
seule fonction d'agrégation, contrairement à ce qu'une version antérieure de cette section indiquait.

**Fonctionnement** :
- `charger_stats_du_jour()` (cellule 12) → totaux du jour, dernières détections
- `charger_statistiques()` (cellule 23) → réutilisée ici uniquement pour `presence_par_heure` (graphique) et
  `personnes` (classées par nombre de détections décroissant, pour le tableau du bas de page)
- `charger_detections_semaine()` → nombre de détections par jour sur les 7 derniers jours (graphique en barres)
- `calculer_etat_systeme()` → caméras en ligne (frames reçues il y a moins de 5 secondes), stockage disque
  (`shutil.disk_usage`), charge CPU/mémoire (`psutil`) — ce sont des métriques **réelles** de la machine qui
  exécute le serveur, pas des exemples statiques

**Le moteur de templates Jinja2** : `render_template("dashboard.html", stats=..., ...)` transmet ces
dictionnaires au moteur Jinja2, qui remplace chaque `{{ stats.total }}` (par exemple) par sa valeur réelle au
moment du rendu. Jinja2 a été créé par Armin Ronacher (le même auteur que Flask) en 2008, en s'inspirant du
système de templates de Django (2005), lui-même héritier d'une longue tradition de moteurs de templates web
remontant aux *Server Side Includes* (SSI) du serveur NCSA HTTPd dans les années 1990.

**Outils et technologies utilisés** :
- **Flask** — `render_template()`
- **Jinja2** (moteur de templates sous-jacent)
- **PyMongo** (via les 3 fonctions `charger_...`)
- **shutil**, **psutil**, **threading** (via `calculer_etat_systeme()`)


In [29]:
def charger_detections_semaine():
    """
    Nombre de détections par jour sur les 7 derniers jours (aujourd'hui inclus),
    pour le graphique "Détections par jour" du tableau de bord.
    """
    jours = []
    for i in range(6, -1, -1):
        d = datetime.now() - timedelta(days=i)
        date_str = d.strftime("%Y-%m-%d")
        total = db.Detections.count_documents({"date": date_str})
        jours.append({"label": d.strftime("%a")[:3].capitalize(), "total": total})
    return jours


def calculer_etat_systeme():
    """
    Indicateurs réels pour la carte "État du système" du tableau de bord :
    caméras en ligne (basé sur les frames reçues récemment), stockage utilisé
    sur le disque, et charge CPU/mémoire de la machine qui exécute le serveur.
    """
    with verrou_frames:
        derniers = dict(dernieres_maj_frames)
    maintenant = datetime.now()
    cameras_en_ligne = sum(
        1 for maj in derniers.values()
        if (maintenant - maj).total_seconds() < 5
    )

    usage_disque = shutil.disk_usage(".")
    stockage_go_utilise = round((usage_disque.total - usage_disque.free) / (1024 ** 3), 1)
    stockage_go_total = round(usage_disque.total / (1024 ** 3), 1)

    return {
        "cameras_en_ligne": cameras_en_ligne,
        "cameras_total": len(CAMERAS),
        "stockage_go_utilise": stockage_go_utilise,
        "stockage_go_total": stockage_go_total,
        "cpu_pourcent": round(psutil.cpu_percent(interval=0.1), 0),
        "memoire_pourcent": round(psutil.virtual_memory().percent, 0),
    }


In [30]:
@app.route("/dashboard")
@require_admin
def dashboard():
    stats = charger_stats_du_jour()
    stats_detail = charger_statistiques()  # réutilisé pour presence_par_heure + personnes
    return render_template(
        "dashboard.html",
        stats=stats,
        presence_par_heure=stats_detail["presence_par_heure"],
        personnes=sorted(stats_detail["personnes"], key=lambda p: p["detections"], reverse=True),
        detections_semaine=charger_detections_semaine(),
        etat_systeme=calculer_etat_systeme(),
    )

### 20. Route `/realtime`

**Rôle** : servir la page de supervision temps réel (vue administrateur). Elle ne transmet aucune donnée calculée
pour l'instant — le flux vidéo est chargé séparément par le navigateur via `/video_feed/<nom_camera>` (cellule 17),
appelé directement depuis la balise `<img>` du template, indépendamment de cette route.

In [31]:
@app.route("/realtime")
@require_admin
def realtime():
    stats = charger_stats_du_jour()
    return render_template(
        "realtime.html",
        dernieres=stats["dernieres"][:5],  # seulement les 5 plus récentes en vue temps réel
        cameras=CAMERAS,
        stats_systeme=calculer_stats_systeme(),
    )

In [32]:
@app.route("/api/dernieres_detections")
@require_admin
def api_dernieres_detections():
    """
    Version JSON de la liste des dernières détections, utilisée par le
    JavaScript de realtime.html pour rafraîchir la barre latérale sans
    recharger toute la page (donc sans interrompre le flux vidéo MJPEG).
    """
    stats = charger_stats_du_jour()
    dernieres = stats["dernieres"][:5]
    payload = [
        {
            "id": l["id"],
            "nom": l["nom"],
            "statut": l["statut"],
            "heure": l["heure"],
            "camera": l["camera"],
            "score_detection": l["score_detection"],
            "image": l.get("image"),
        }
        for l in dernieres
    ]
    return jsonify(payload)


### 21. Route `/vue_users`

**Rôle** : la même logique que `/realtime`, mais pour la vue destinée aux employés (moins d'informations
administratives affichées). `nom_camera="CAM-01"` est transmis en dur pour l'instant — une seule caméra affichée
par page ; un sélecteur multi-caméras pourra être ajouté plus tard si nécessaire.

In [33]:
@app.route("/vue_users")
@require_employe
def vue_users():
    stats = charger_stats_du_jour()
    return render_template(
        "vue_users.html",
        dernieres=stats["dernieres"][:5],  # seulement les 5 plus récentes en vue temps réel
        cameras=CAMERAS,
        stats_systeme=calculer_stats_systeme(),
    )

### 22. `charger_statistiques()` — agrégations pour la page Statistiques

**Rôle** : calculer trois choses pour `statistics.html` : la répartition des détections par heure, le taux global
de reconnaissance, et le détail par personne (avec rôle/département).

**Principe — l'histogramme par heure** : `presence_par_heure` est un **histogramme** au sens statistique classique
(le terme a été proposé par Karl Pearson en 1895) : on découpe la journée en 24 catégories (les heures), et on
compte combien de détections tombent dans chaque catégorie. C'est la structure de données la plus simple pour
visualiser une distribution de fréquence dans le temps — exactement ce qu'affichent les barres du graphique
« Présence par heure » de votre page HTML.

**Le taux de reconnaissance** est un simple ratio (`connus / total × 100`), une proportion — rien de plus qu'une
règle de trois, mais c'est la métrique la plus lisible pour un humain qui veut juger la performance globale du
système sans lire un journal détaillé.

**La jointure manuelle** (`db.Persons.find({"nom": {"$in": noms}}, ...)`) : MongoDB, en tant que base
orientée documents, n'a historiquement pas de jointure native aussi simple qu'un `JOIN` SQL (les données sont
plutôt censées être dénormalisées, c'est-à-dire dupliquées dans chaque document pour éviter d'avoir à joindre).
Ici on fait le lien « à la main » entre deux collections (`Detections` et `Persons`) en récupérant les
fiches correspondantes via l'opérateur `$in` (« la valeur du champ `nom` doit être l'une de ces valeurs de la
liste »), puis en construisant un dictionnaire Python pour un accès rapide (`infos_par_nom.get(nom, {})`) — une
solution simple, adaptée au faible volume de personnes différentes par jour dans ce contexte.

In [34]:
def charger_statistiques(date=None):
    """
    Agrège les données pour la page statistics : présence par heure,
    taux de reconnaissance, et détail par personne (avec poste/département,
    la liste complète de ses heures de passage, et une photo prise lors
    de sa détection la plus récente).
    """
    date = date or datetime.now().strftime("%Y-%m-%d")
    logs = list(db.Detections.find({"date": date}).sort("heure", 1))

    # Présence par heure (0h à 23h)
    presence_par_heure = [0] * 24
    for l in logs:
        heure = int(l["heure"].split(":")[0])
        presence_par_heure[heure] += 1

    total = len(logs)
    connus = sum(1 for l in logs if l["statut"] == "SUCCES")
    inconnus = total - connus
    taux_reconnaissance = round((connus / total) * 100, 1) if total else 0.0

    # Détail par personne connue : toutes les heures de passage, nombre de
    # passages, et une photo (celle de la détection la plus récente, les
    # logs étant triés par heure croissante ci-dessus).
    personnes = {}
    for l in logs:
        if l["statut"] != "SUCCES":
            continue
        nom = l["nom"]
        if nom not in personnes:
            personnes[nom] = {
                "nom": nom,
                "premiere": l["heure"],
                "derniere": l["heure"],
                "detections": 0,
                "heures": [],
                "photo": None,
            }
        personnes[nom]["derniere"] = l["heure"]
        personnes[nom]["detections"] += 1
        personnes[nom]["heures"].append(l["heure"])
        if l.get("image"):
            personnes[nom]["photo"] = l["image"]

    # Jointure avec personnes_connues pour poste/département
    noms = list(personnes.keys())
    fiches = db.Persons.find({"nom": {"$in": noms}}, {"nom": 1, "role": 1, "departement": 1})
    infos_par_nom = {f["nom"]: f for f in fiches}

    for nom, p in personnes.items():
        info = infos_par_nom.get(nom, {})
        p["role"] = info.get("role", "")
        p["departement"] = info.get("departement", "")

    return {
        "date": date,
        "total": total,
        "connus": connus,
        "inconnus": inconnus,
        "taux_reconnaissance": taux_reconnaissance,
        "presence_par_heure": presence_par_heure,
        "personnes": list(personnes.values()),
    }

### 23. Route `/statistics`

**Rôle** : relier `charger_statistiques()` au template `statistics.html`, selon le même principe que la route
`/dashboard` (cellule 19).

In [35]:
@app.route("/statistics")
@require_admin
def statistics():
    stats = charger_statistiques()
    return render_template("statistics.html", stats=stats)

### 23.1 Génération des exports — Excel et PDF

**Rôle** : deux fonctions génériques, réutilisées par les 3 pages exportables (Inconnus, Personnes connues,
Détections) — un seul endroit pour générer chaque format, pas une implémentation par page.

**Fonctionnement** :
- `generer_reponse_excel()` construit un classeur avec **openpyxl**, écrit en mémoire (`io.BytesIO`, pas de
  fichier temporaire sur disque), puis le renvoie comme pièce jointe téléchargeable.
- `generer_reponse_pdf()` construit un document avec **ReportLab** (`SimpleDocTemplate` + `Table`), en
  orientation paysage pour laisser de la place aux colonnes, avec un en-tête de tableau coloré et des lignes
  alternées pour la lisibilité.
- Dans les deux cas, l'en-tête HTTP `Content-Disposition: attachment` indique au navigateur de proposer un
  téléchargement plutôt que d'afficher le contenu.

**À installer** (bibliothèques tierces, pas incluses avec Python) :
```
pip install openpyxl reportlab
```

**Outils et technologies utilisés** :
- **openpyxl** — génération de fichiers `.xlsx` (format Excel natif, basé sur XML/ZIP)
- **ReportLab** — génération de fichiers `.pdf`
- **io** (bibliothèque standard) — tampon mémoire (`BytesIO`)
- **Flask** — `Response`, en-têtes HTTP personnalisés

In [36]:
def generer_reponse_excel(entetes, lignes, nom_fichier, chemin_logo=None, chemins_photos=None):
    """
    Construit un classeur Excel (.xlsx) téléchargeable : logo de l'entreprise en
    en-tête, colonnes d'en-tête stylées (fond bleu, texte blanc), et — si
    chemins_photos est fourni (une entrée par ligne, un chemin ou None) — une
    miniature photo en première colonne de chaque ligne.
    """
    classeur = Workbook()
    feuille = classeur.active

    ligne_entete = 1
    if chemin_logo and os.path.exists(chemin_logo):
        logo = ImageExcel(chemin_logo)
        logo.height, logo.width = 60, 60
        feuille.add_image(logo, "A1")
        feuille.row_dimensions[1].height = 46
        ligne_entete = 4  # laisse la place au logo avant la ligne d'en-tête

    entetes_completes = (["Photo"] if chemins_photos else []) + entetes
    for col_idx, texte in enumerate(entetes_completes, start=1):
        cellule = feuille.cell(row=ligne_entete, column=col_idx, value=texte)
        cellule.font = Font(bold=True, color="FFFFFF")
        cellule.fill = PatternFill("solid", fgColor="2563EB")
        cellule.alignment = Alignment(horizontal="center")
        feuille.column_dimensions[cellule.column_letter].width = 20

    for i, ligne in enumerate(lignes):
        r = ligne_entete + 1 + i
        decalage = 0
        if chemins_photos:
            chemin_photo = chemins_photos[i]
            if chemin_photo and os.path.exists(chemin_photo):
                miniature = ImageExcel(chemin_photo)
                miniature.height, miniature.width = 40, 40
                feuille.add_image(miniature, f"A{r}")
            feuille.row_dimensions[r].height = 32
            decalage = 1
        for col_idx, valeur in enumerate(ligne, start=1 + decalage):
            feuille.cell(row=r, column=col_idx, value=valeur)

    tampon = io.BytesIO()
    classeur.save(tampon)
    tampon.seek(0)

    reponse = Response(
        tampon.getvalue(),
        mimetype="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
    )
    reponse.headers["Content-Disposition"] = f"attachment; filename={nom_fichier}"
    return reponse


# Couleurs reprises telles quelles des variables CSS du site (style.css),
# pour que le PDF exporté ait le même habillage visuel que l'interface web.
COULEUR_PRIMAIRE = colors.HexColor("#004ac6")     # var(--primary)
COULEUR_SUR_PRIMAIRE = colors.white                # var(--on-primary)
COULEUR_LIGNE_ALT = colors.HexColor("#f3f3fe")      # var(--surface-container-low)
COULEUR_BORDURE = colors.HexColor("#c3c6d7")        # var(--outline-variant)
COULEUR_TEXTE = colors.HexColor("#191b23")          # var(--on-surface)
COULEUR_TEXTE_DISCRET = colors.HexColor("#434655")  # var(--on-surface-variant)

MARGE_PDF = 18  # points ; marges réduites pour que le tableau occupe toute la largeur de la page


def generer_reponse_pdf(titre, entetes, lignes, nom_fichier, chemin_logo=None, chemins_photos=None):
    """
    Construit un document PDF téléchargeable (tableau paysage), avec le même
    habillage visuel que le site (bandeau bleu --primary, mêmes couleurs de
    bordures/lignes alternées) et un tableau étiré sur toute la largeur
    imprimable de la page — pas une petite table perdue sur une page blanche.
    Si chemins_photos est fourni (une entrée par ligne, un chemin ou None),
    une miniature photo est insérée en première colonne de chaque ligne.
    """
    tampon = io.BytesIO()
    largeur_page, hauteur_page = landscape(A4)
    document = SimpleDocTemplate(
        tampon, pagesize=landscape(A4),
        leftMargin=MARGE_PDF, rightMargin=MARGE_PDF, topMargin=MARGE_PDF, bottomMargin=MARGE_PDF,
    )
    largeur_utile = largeur_page - 2 * MARGE_PDF
    styles = getSampleStyleSheet()

    elements = []

    # En-tête (logo + titre) en noir sur blanc — sur toute la largeur utile, sans
    # bandeau de couleur (contrairement au tableau, qui lui reste en --primary).
    style_titre_bandeau = ParagraphStyle(
        "TitreBandeau", parent=styles["Title"], textColor=COULEUR_TEXTE,
        alignment=1, fontSize=18,
    )
    cellule_logo = ""
    if chemin_logo and os.path.exists(chemin_logo):
        cellule_logo = ImagePdf(chemin_logo, width=36, height=36)
    bandeau = Table(
        [[cellule_logo, Paragraph(titre, style_titre_bandeau), ""]],
        colWidths=[50, largeur_utile - 100, 50],
    )
    bandeau.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), colors.white),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (0, 0), (0, 0), "CENTER"),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
    ]))
    elements.append(bandeau)
    elements.append(Spacer(1, 4))
    elements.append(Paragraph(
        f"Généré le {datetime.now().strftime('%Y-%m-%d à %H:%M')}",
        ParagraphStyle("SousTitre", parent=styles["Normal"], textColor=COULEUR_TEXTE_DISCRET, fontSize=8),
    ))
    elements.append(Spacer(1, 10))

    entetes_completes = (["Photo"] if chemins_photos else []) + entetes
    lignes_pdf = []
    for i, ligne in enumerate(lignes):
        ligne_pdf = list(ligne)
        if chemins_photos:
            chemin_photo = chemins_photos[i]
            if chemin_photo and os.path.exists(chemin_photo):
                cellule_photo = ImagePdf(chemin_photo, width=30, height=30)
            else:
                cellule_photo = ""
            ligne_pdf = [cellule_photo] + ligne_pdf
        lignes_pdf.append(ligne_pdf)

    # Répartition des colonnes sur toute la largeur utile : la colonne Photo a une
    # largeur fixe (juste assez pour la miniature), les autres colonnes se
    # partagent équitablement le reste — le tableau remplit donc toujours toute
    # la largeur de la page, quel que soit le nombre de colonnes.
    n_colonnes = len(entetes_completes)
    if chemins_photos:
        largeur_photo = 40
        largeur_colonne = (largeur_utile - largeur_photo) / (n_colonnes - 1)
        largeurs_colonnes = [largeur_photo] + [largeur_colonne] * (n_colonnes - 1)
    else:
        largeurs_colonnes = [largeur_utile / n_colonnes] * n_colonnes

    donnees_tableau = [entetes_completes] + lignes_pdf
    tableau = Table(donnees_tableau, colWidths=largeurs_colonnes, repeatRows=1)
    tableau.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), COULEUR_PRIMAIRE),
        ("TEXTCOLOR", (0, 0), (-1, 0), COULEUR_SUR_PRIMAIRE),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("GRID", (0, 0), (-1, -1), 0.5, COULEUR_BORDURE),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("TEXTCOLOR", (0, 1), (-1, -1), COULEUR_TEXTE),
        ("BACKGROUND", (0, 1), (-1, -1), colors.white),
    ]))
    elements.append(tableau)

    def pied_de_page(canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(COULEUR_TEXTE_DISCRET)
        canvas.drawString(MARGE_PDF, 10, "Système de surveillance")
        canvas.drawRightString(largeur_page - MARGE_PDF, 10, f"Page {doc.page}")
        canvas.restoreState()

    document.build(elements, onFirstPage=pied_de_page, onLaterPages=pied_de_page)

    tampon.seek(0)
    reponse = Response(tampon.getvalue(), mimetype="application/pdf")
    reponse.headers["Content-Disposition"] = f"attachment; filename={nom_fichier}"
    return reponse

### 24. Route `/unknowns`

**Rôle** : afficher la liste des inconnus en attente d'identification, avec filtrage.

**Pourquoi convertir `_id` en chaîne ?** Chaque document MongoDB possède un champ `_id` de type **ObjectId** —
un identifiant binaire de 12 octets (4 octets d'horodatage + 5 octets aléatoires générés une fois par processus +
3 octets de compteur incrémental), conçu par les ingénieurs de MongoDB pour garantir l'unicité *sans coordination
centrale*, contrairement aux identifiants auto-incrémentés des bases SQL classiques qui nécessitent un point de
synchronisation unique. Le moteur de templates Jinja2 ne sait afficher que du texte simple : `str(doc["_id"])`
convertit cet identifiant binaire en sa représentation textuelle hexadécimale, utilisable dans un champ caché du
formulaire HTML (voir cellule 27, où cette valeur revient sous forme de texte).

**Filtrage** — deux éléments indépendants, non combinés :
- `periode` : plage rapide ("hier", "3j", "7j", "10j", "1mois", "2mois"), traduite en comparaison sur `date`
  (un vrai objet `datetime` dans `UnknownPersons`, contrairement à `Detections` qui stocke une chaîne)
- `recherche` : texte libre — une date au format `YYYY-MM-DD` (détectée via une expression régulière) filtre
  sur le jour entier ; tout autre texte filtre par caméra, via une recherche partielle insensible à la casse
  (opérateur MongoDB `$regex`/`$options: "i"`)

**Outils et technologies utilisés** :
- **Flask** — `request.args`, `render_template()`
- **PyMongo** — `db.UnknownPersons.find()` avec `$gte`/`$lt`/`$regex`
- **re** (bibliothèque standard) — détection du format date, `re.escape()` pour échapper les caractères spéciaux
- **datetime** / **timedelta** — calcul des plages de dates


In [37]:
def construire_requete_unknowns(periode, recherche):
    """Construit le filtre MongoDB pour UnknownPersons, réutilisé par la page et son export CSV."""
    PERIODES = {"hier": 1, "3j": 3, "7j": 7, "10j": 10, "1mois": 30, "2mois": 60}

    requete = {"traite": False}

    if periode in PERIODES:
        requete["date"] = {"$gte": datetime.now() - timedelta(days=PERIODES[periode])}

    if recherche:
        if re.match(r"^\d{4}-\d{2}-\d{2}$", recherche):
            debut_jour = datetime.strptime(recherche, "%Y-%m-%d")
            requete["date"] = {"$gte": debut_jour, "$lt": debut_jour + timedelta(days=1)}
        else:
            requete["camera"] = {"$regex": re.escape(recherche), "$options": "i"}

    return requete


@app.route("/unknowns")
@require_admin
def unknowns():
    """
    Filtrage — deux éléments indépendants, non combinés :
    - periode : plage rapide ("hier", "3j", "7j", "10j", "1mois", "2mois")
    - recherche : texte libre — une date (YYYY-MM-DD) filtre par jour, tout
      autre texte filtre par caméra (recherche partielle, insensible à la casse)
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()

    requete = construire_requete_unknowns(periode, recherche)
    inconnus = list(db.UnknownPersons.find(requete).sort("date", -1))
    # Conversion de l'ObjectId en chaîne pour l'utiliser dans les formulaires du template
    for doc in inconnus:
        doc["id"] = str(doc["_id"])

    return render_template(
        "unknowns.html",
        inconnus=inconnus,
        filtres={"periode": periode, "recherche": recherche},
    )


@app.route("/unknowns/export")
@require_admin
def export_unknowns():
    """
    Exporte, au format choisi (xlsx ou pdf), exactement les mêmes inconnus que
    ceux affichés (mêmes filtres), avec les mêmes colonnes que la page (dont
    "Heures de passage", voir enregistrer_ou_regrouper_inconnu) et la photo
    de chaque fiche.
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()
    format_export = request.args.get("format", "xlsx")

    requete = construire_requete_unknowns(periode, recherche)
    inconnus = list(db.UnknownPersons.find(requete).sort("date", -1))

    entetes = ["Date", "Heures de passage", "Camera", "Confiance"]
    lignes = []
    chemins_photos = []
    for doc in inconnus:
        lignes.append([
            doc["date"].strftime("%Y-%m-%d"),
            ", ".join(doc.get("heures") or [doc["date"].strftime("%H:%M:%S")]),
            doc.get("camera", ""),
            f"{round(doc.get('score', 0) * 100, 2)}%",
        ])
        image = doc.get("image")
        chemins_photos.append(os.path.join(DOSSIER_INCONNUS, image) if image else None)

    if format_export == "pdf":
        return generer_reponse_pdf(
            "Personnes inconnues", entetes, lignes, "personnes_inconnues.pdf",
            chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
        )
    return generer_reponse_excel(
        entetes, lignes, "personnes_inconnues.xlsx",
        chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
    )

### 25. `identifier_inconnu()` — transformer un inconnu en personne connue

**Rôle** : quand un administrateur identifie manuellement un inconnu via le formulaire, cette fonction relit
l'image sauvegardée, y recalcule un embedding, l'ajoute à `personnes_connues`, et marque la fiche comme traitée.

**Pourquoi recalculer l'embedding plutôt que le réutiliser ?** Au moment de la détection initiale (cellule 16), on
a choisi de ne sauvegarder que l'*image* du visage, pas son embedding — un choix qui simplifie le flux principal
(pas besoin de conserver un vecteur de 512 flottants en attente), au prix de devoir refaire tourner les deux
modèles (détection + reconnaissance) une seconde fois sur l'image, une fois l'identité confirmée. Pour le faible
volume de fiches à traiter manuellement, ce recalcul reste négligeable en temps de calcul.

**`ObjectId(detection_id)`** reconstruit l'identifiant binaire MongoDB à partir de sa représentation textuelle
reçue du formulaire — l'opération inverse exacte de `str(doc["_id"])` faite en cellule 24.

In [38]:
def identifier_inconnu(detection_id, nom, role, departement):
    """
    Recalcule l'embedding à partir de l'image sauvegardée d'un inconnu,
    l'ajoute à personnes_connues, et marque la fiche comme traitée.
    """
    from bson import ObjectId

    doc = db.UnknownPersons.find_one({"_id": ObjectId(detection_id)})
    if doc is None:
        print(f"Fiche inconnue introuvable : {detection_id}")
        return False

    chemin_image = os.path.join(DOSSIER_INCONNUS, doc["image"])
    img = cv2.imread(chemin_image)
    if img is None:
        print(f"Impossible de relire l'image : {chemin_image}")
        return False

    bboxes, kpss = det_model.detect(img, max_num=0, metric="default")
    if len(bboxes) == 0:
        print(f"Aucun visage retrouvé dans l'image sauvegardée : {chemin_image}")
        return False

    bbox = bboxes[0, :4]
    face = Face(bbox=bbox, kps=kpss[0], det_score=bboxes[0, 4])
    rec_model.get(img, face)

    if not hasattr(face, "normed_embedding"):
        print("Échec du recalcul de l'embedding.")
        return False

    db.Persons.insert_one({
        "nom": nom,
        "role": role,
        "departement": departement,
        "embedding": face.normed_embedding.tolist(),
    })

    db.UnknownPersons.update_one(
        {"_id": ObjectId(detection_id)},
        {"$set": {"traite": True}}
    )

    return True

### 26. Route `/identifier` — traitement du formulaire

**Rôle** : recevoir la soumission du formulaire d'identification (méthode HTTP `POST`), appeler
`identifier_inconnu()`, puis recharger la base d'embeddings en mémoire pour que la reconnaissance en direct
(`/video_feed`) prenne immédiatement en compte la nouvelle personne.

**GET vs POST** : ces deux verbes du protocole HTTP (normalisés dès la version 1.0 de HTTP, 1996) ont des
sémantiques différentes — `GET` est censé être *sans effet de bord* (consulter une page), `POST` est destiné à
*modifier un état* (ici, créer une nouvelle personne connue). Utiliser `POST` pour ce formulaire respecte cette
convention et évite, par exemple, qu'un navigateur ne réexécute accidentellement l'identification simplement en
rafraîchissant la page.

**Le mot-clé `global`** : sans lui, l'affectation `known_embeddings, known_names = ...` à l'intérieur de la
fonction créerait des variables *locales* à cette fonction, sans effet sur les variables du même nom utilisées par
`generer_frames()` ailleurs dans le notebook — une subtilité classique de la portée des variables en Python, où
une affectation dans une fonction est locale par défaut, sauf déclaration explicite du contraire.

In [39]:
@app.route("/identifier", methods=["POST"])
@require_admin
def identifier():
    global known_embeddings, known_names

    detection_id = request.form.get("detection_id")
    nom = request.form.get("nom")
    role = request.form.get("poste", "")  # nom du champ HTML inchangé, valeur stockée sous "role"
    departement = request.form.get("departement", "")

    succes = identifier_inconnu(detection_id, nom, role, departement)
    if succes:
        known_embeddings, known_names = charger_embeddings_mongo()
    else:
        print("Identification échouée.")

    return redirect(url_for("unknowns"))

### 27. Route `/personnes_connues`

**Rôle** : afficher une ligne par personne réellement reconnue par la caméra (pas une ligne par embedding —
`Persons` en contient plusieurs dizaines par personne), avec filtrage.

**Fonctionnement** : contrairement au dashboard (qui utilise une fonction d'agrégation dédiée), cette route
construit directement sa liste dans le corps de la vue : elle part des noms distincts ayant au moins une
détection réussie (`db.Detections.distinct("nom", {"statut": "SUCCES"})`), puis, pour chacun, va chercher sa
fiche d'identité (`role`, `departement`) dans `Persons` et sa détection la plus récente dans `Detections` — dont
l'image sert de photo de profil affichée (`Persons` ne stocke que des embeddings, aucune photo).

**Filtrage** — deux éléments indépendants, non combinés :
- `periode` : plage rapide, appliquée sur la date de la détection la plus récente de chaque personne
- `recherche` : texte libre — une date filtre par date exacte, tout autre texte filtre par nom (recherche
  partielle insensible à la casse, faite en Python sur la chaîne déjà en mémoire, pas via MongoDB)

**Outils et technologies utilisés** :
- **Flask** — `request.args`, `render_template()`
- **PyMongo** — `distinct()`, `find_one()` avec tri (`sort`)
- **re** (bibliothèque standard) — détection du format date
- **datetime** / **timedelta** — calcul des plages de dates
- Tri Python pur (`sorted`/`.sort()` avec clé composite date+heure)


In [40]:
def obtenir_personnes_connues(periode, recherche):
    """
    Persons contient un document par PHOTO (donc plusieurs dizaines par personne,
    un par embedding extrait). Cette fonction regroupe par nom pour ne retourner
    qu'une ligne par personne, enrichie avec sa dernière détection réelle par la
    caméra (Detections) — réutilisée par la page et son export CSV.
    """
    PERIODES = {"hier": 1, "3j": 3, "7j": 7, "10j": 10, "1mois": 30, "2mois": 60}

    date_limite = None
    if periode in PERIODES:
        date_limite = (datetime.now() - timedelta(days=PERIODES[periode])).strftime("%Y-%m-%d")

    date_recherchee = None
    nom_recherche = None
    if recherche:
        if re.match(r"^\d{4}-\d{2}-\d{2}$", recherche):
            date_recherchee = recherche
        else:
            nom_recherche = recherche.lower()

    noms_deja_detectes = db.Detections.distinct("nom", {"statut": "SUCCES"})
    personnes = []

    for nom in noms_deja_detectes:
        if nom_recherche and nom_recherche not in nom.lower():
            continue

        fiche = db.Persons.find_one({"nom": nom})
        if fiche is None:
            continue  # nom présent dans Detections mais plus dans Persons (supprimé entretemps)

        requete_detection = {"nom": nom, "statut": "SUCCES"}
        if date_recherchee:
            requete_detection["date"] = date_recherchee
        elif date_limite:
            requete_detection["date"] = {"$gte": date_limite}

        derniere_detection = db.Detections.find_one(
            requete_detection,
            sort=[("date", -1), ("heure", -1)],
        )
        if derniere_detection is None:
            continue  # aucune détection ne correspond à la période/date recherchée

        # La fiche Persons ne stocke pas de photo (seulement les embeddings) ;
        # on utilise donc l'image capturée lors de la détection la plus
        # récente de cette personne comme photo de profil affichée.
        personnes.append({
            "id": str(fiche["_id"]),
            "nom": nom,
            "role": fiche.get("role"),
            "departement": fiche.get("departement"),
            "photo": derniere_detection.get("image"),
            "date_ajout": f"{derniere_detection['date']} - {derniere_detection['heure']}",
            "_tri": (derniere_detection["date"], derniere_detection["heure"]),
            "statut": "Actif",
        })

    personnes.sort(key=lambda p: p["_tri"], reverse=True)  # plus récente détection en premier
    for p in personnes:
        del p["_tri"]

    return personnes


@app.route("/personnes_connues")
@require_admin
def personnes_connues():
    """
    Filtrage — deux éléments indépendants, non combinés :
    - periode : plage rapide ("hier", "3j", "7j", "10j", "1mois", "2mois")
    - recherche : texte libre — une date au format YYYY-MM-DD filtre par date,
      tout autre texte filtre par nom (recherche partielle, insensible à la casse)
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()
    personnes = obtenir_personnes_connues(periode, recherche)

    return render_template(
        "personnesconnues.html",
        personnes=personnes,
        filtres={"periode": periode, "recherche": recherche},
    )


@app.route("/personnes_connues/export")
@require_admin
def export_personnes_connues():
    """
    Exporte, au format choisi (xlsx ou pdf), exactement les mêmes personnes que
    celles affichées (mêmes filtres), avec les mêmes colonnes que la page et la
    photo de chacune.
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()
    format_export = request.args.get("format", "xlsx")
    personnes = obtenir_personnes_connues(periode, recherche)

    entetes = ["Nom", "Role", "Departement", "Derniere detection", "Statut"]
    lignes = [
        [p["nom"], p.get("role") or "", p.get("departement") or "", p["date_ajout"], p["statut"]]
        for p in personnes
    ]
    chemins_photos = [
        os.path.join(DOSSIER_SUCCES, p["photo"]) if p.get("photo") else None
        for p in personnes
    ]

    if format_export == "pdf":
        return generer_reponse_pdf(
            "Personnes connues", entetes, lignes, "personnes_connues.pdf",
            chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
        )
    return generer_reponse_excel(
        entetes, lignes, "personnes_connues.xlsx",
        chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
    )

In [41]:
# ### 27.1 Route `/modifier_personne` — édition du rôle/département d'un employé
#
# **Rôle** : traiter la soumission du formulaire d'édition ouvert depuis `personnesconnues.html`.
#
# **Pourquoi `update_many` et pas `update_one` ?** `Persons` contient un document par PHOTO
# d'enrôlement (un par embedding), donc plusieurs dizaines de documents pour une même personne
# (voir `extract_face_embeddings`, cellule 9). Modifier uniquement le document ciblé par son
# `_id` laisserait le rôle/département incohérent entre les différentes photos de la même
# personne ; `update_many` sur le champ `nom` met à jour toutes ses fiches en une seule fois.

@app.route("/modifier_personne", methods=["POST"])
@require_admin
def modifier_personne():
    nom = request.form.get("nom", "").strip()
    role = request.form.get("role", "").strip()
    departement = request.form.get("departement", "").strip()

    if nom:
        db.Persons.update_many(
            {"nom": nom},
            {"$set": {"role": role, "departement": departement}},
        )

    return redirect(url_for("personnes_connues"))


### 28. Route `/detections`

**Rôle** : afficher le journal des détections (`dernieres_detections.html`), avec filtrage. Sans filtre,
n'affiche que les détections d'**aujourd'hui** (comportement par défaut) — l'option "Tout" du menu période
retire explicitement cette restriction.

**Filtrage** — deux éléments indépendants, non combinés :
- `periode` : plage rapide, ou "tout" pour lever toute restriction de date
- `recherche` : texte libre — une date filtre par date exacte ; tout autre texte filtre par **nom OU caméra**
  simultanément, via l'opérateur MongoDB `$or` combiné à deux `$regex`

**Outils et technologies utilisés** :
- **Flask** — `request.args`, `render_template()`
- **PyMongo** — `db.Detections.find()` avec `$gte`, `$or`, `$regex`
- **re** (bibliothèque standard) — détection du format date, `re.escape()`
- **datetime** / **timedelta** — calcul des plages de dates


In [42]:
def construire_requete_detections(periode, recherche):
    """
    Construit le filtre MongoDB pour Detections, réutilisé par la page et son
    export CSV. Sans filtre, restreint à aujourd'hui (comportement par défaut) —
    periode="tout" lève explicitement cette restriction.
    """
    PERIODES = {"hier": 1, "3j": 3, "7j": 7, "10j": 10, "1mois": 30, "2mois": 60}

    requete = {}
    date_filtree = False

    if periode == "tout":
        date_filtree = True  # aucune restriction de date appliquée
    elif periode in PERIODES:
        limite = (datetime.now() - timedelta(days=PERIODES[periode])).strftime("%Y-%m-%d")
        requete["date"] = {"$gte": limite}
        date_filtree = True

    if recherche:
        if re.match(r"^\d{4}-\d{2}-\d{2}$", recherche):
            requete["date"] = recherche
            date_filtree = True
        else:
            motif = {"$regex": re.escape(recherche), "$options": "i"}
            requete["$or"] = [{"nom": motif}, {"camera": motif}]

    if not date_filtree:
        requete["date"] = datetime.now().strftime("%Y-%m-%d")

    return requete


@app.route("/detections")
@require_admin
def detections():
    """
    Filtrage — deux éléments indépendants, non combinés :
    - periode : plage rapide ("hier", "3j", "7j", "10j", "1mois", "2mois", "tout")
    - recherche : texte libre — une date (YYYY-MM-DD) filtre par jour, tout
      autre texte filtre par nom OU par caméra (recherche partielle)
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()

    requete = construire_requete_detections(periode, recherche)
    dernieres = list(db.Detections.find(requete).sort([("date", -1), ("heure", -1)]))
    for l in dernieres:
        l["id"] = str(l["_id"])

    return render_template(
        "dernieres_detections.html",
        dernieres=dernieres,
        filtres={"periode": periode, "recherche": recherche},
    )


@app.route("/detections/export")
@require_admin
def export_detections():
    """
    Exporte, au format choisi (xlsx ou pdf), exactement les mêmes détections que
    celles affichées (mêmes filtres), avec les mêmes colonnes que la page et la
    photo de chaque détection (dossier succes/ ou inconnus/ selon le statut,
    comme dans le template).
    """
    periode = request.args.get("periode", "").strip()
    recherche = request.args.get("recherche", "").strip()
    format_export = request.args.get("format", "xlsx")

    requete = construire_requete_detections(periode, recherche)
    dernieres = list(db.Detections.find(requete).sort([("date", -1), ("heure", -1)]))

    entetes = ["Date", "Heure", "Nom ou Statut", "Camera", "Confiance"]
    lignes = []
    chemins_photos = []
    for l in dernieres:
        lignes.append([
            l["date"],
            l["heure"],
            l["nom"] if l["statut"] == "SUCCES" else "Inconnu",
            l.get("camera", ""),
            f"{round(l.get('score_detection', 0) * 100, 2)}%",
        ])
        image = l.get("image")
        dossier = DOSSIER_INCONNUS if l["statut"] == "REFUSE" else DOSSIER_SUCCES
        chemins_photos.append(os.path.join(dossier, image) if image else None)

    if format_export == "pdf":
        return generer_reponse_pdf(
            "Détections", entetes, lignes, "detections.pdf",
            chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
        )
    return generer_reponse_excel(
        entetes, lignes, "detections.xlsx",
        chemin_logo=CHEMIN_LOGO, chemins_photos=chemins_photos,
    )


### 29. Stockage des administrateurs (collection Users)

**Rôle** : gérer les comptes admin dans `Users`, avec un **mot de passe unique partagé par tous les administrateurs**
(pas un mot de passe par personne). Chaque admin a son propre document (juste un nom, pour l'identifier), et un
document séparé (`{"type": "mot_de_passe_admin", ...}`) stocke le hash du mot de passe commun. Se connecter demande
donc un nom qui existe bien parmi les admins **et** le mot de passe partagé.

**Important** : au tout premier lancement, un email admin par défaut et un mot de passe commun par défaut
(`00000000`) sont créés automatiquement — à changer immédiatement depuis la page Paramètres.

In [43]:
from werkzeug.security import generate_password_hash, check_password_hash


def charger_mot_de_passe_admin():
    """
    Charge le mot de passe commun à tous les administrateurs (un seul document,
    identifié par type="mot_de_passe_admin"), ou le crée avec la valeur par défaut.
    """
    doc = db.Users.find_one({"type": "mot_de_passe_admin"})
    if doc is not None:
        return doc

    doc_defaut = {
        "type": "mot_de_passe_admin",
        "password_hash": generate_password_hash("00000000"),
    }
    db.Users.insert_one(doc_defaut)
    print("Mot de passe admin commun créé (valeur par défaut : '00000000' — à changer dans Paramètres).")
    return doc_defaut


def sauvegarder_mot_de_passe_admin(doc):
    db.Users.update_one({"_id": doc["_id"]}, {"$set": {"password_hash": doc["password_hash"]}})


def creer_admin(nom):
    """Autorise un nom supplémentaire à se connecter en tant qu'administrateur
    (avec le mot de passe commun — pas de mot de passe individuel)."""
    if db.Users.find_one({"nom": nom, "role": "admin"}) is None:
        db.Users.insert_one({"nom": nom, "role": "admin"})
        print(f"Admin ajouté : {nom}")


MOT_DE_PASSE_ADMIN = charger_mot_de_passe_admin()

# S'assure qu'au moins un admin existe, au tout premier lancement
if db.Users.count_documents({"role": "admin"}) == 0:
    creer_admin("Admin")

### 30. Route `/parametres`

**Rôle** : afficher la page Paramètres (`parametres.html`), avec la liste actuelle des caméras.

In [44]:
@app.route("/parametres")
@require_admin
def parametres():
    return render_template("parametres.html", cameras=CAMERAS)

### 31. Route `/parametres/mot_de_passe`

**Rôle** : traiter le formulaire de changement de mot de passe. Vérifie l'ancien mot de passe avec `check_password_hash`, exige une confirmation, puis réécrit `data/admin.json`.

In [45]:
@app.route("/parametres/mot_de_passe", methods=["POST"])
@require_admin
def parametres_mot_de_passe():
    ancien = request.form.get("ancien_mdp", "")
    nouveau = request.form.get("nouveau_mdp", "")
    confirmation = request.form.get("confirmation_mdp", "")

    erreur_mdp = None
    succes_mdp = None

    if not check_password_hash(MOT_DE_PASSE_ADMIN["password_hash"], ancien):
        erreur_mdp = "Mot de passe actuel incorrect."
    elif nouveau != confirmation:
        erreur_mdp = "La confirmation ne correspond pas au nouveau mot de passe."
    elif len(nouveau) < 8:
        erreur_mdp = "Le nouveau mot de passe doit faire au moins 8 caractères."
    else:
        MOT_DE_PASSE_ADMIN["password_hash"] = generate_password_hash(nouveau)
        sauvegarder_mot_de_passe_admin(MOT_DE_PASSE_ADMIN)
        succes_mdp = "Mot de passe mis à jour avec succès pour tous les administrateurs."

    return render_template("parametres.html", cameras=CAMERAS, erreur_mdp=erreur_mdp, succes_mdp=succes_mdp)

### 33. Route `/parametres/camera/supprimer/<nom_camera>`

**Rôle** : retirer une caméra de `CAMERAS` et persister le changement.

In [46]:
@app.route("/parametres/camera/supprimer/<nom_camera>", methods=["POST"])
@require_admin
def parametres_supprimer_camera(nom_camera):
    CAMERAS.pop(nom_camera, None)
    sauvegarder_cameras(CAMERAS)
    return redirect(url_for("parametres"))

### 34. Chargement initial des embeddings connus

**Rôle** : charger une seule fois, avant de démarrer le serveur, les embeddings depuis MongoDB — ce sont ces
variables globales que `generer_frames()` (cellule 16) et la route `/identifier` (cellule 26) utilisent et mettent
à jour ensuite.

In [47]:
known_embeddings, known_names = charger_embeddings_mongo()
if known_embeddings is None:
    print("Aucun embedding en base — lancez extract_face_embeddings() ou identifiez des inconnus d'abord.")

### 35. Lancement du serveur Flask

**Rôle** : démarrer le serveur de développement intégré à Flask, qui écoute désormais les requêtes HTTP.

**Fonctionnement des paramètres** :
- `host="0.0.0.0"` : écoute sur toutes les interfaces réseau de la machine (pas seulement `localhost`), ce qui
  permet d'accéder au serveur depuis un autre appareil du réseau local (utile pour tester depuis un téléphone,
  par exemple, en plus de la connexion DroidCam elle-même)
- `debug=True` : active le mode debug de Flask, qui affiche des pages d'erreur détaillées en cas d'exception
- `use_reloader=False` : **indispensable dans un notebook**. Le *reloader* de Flask, en temps normal, surveille
  les fichiers source et relance tout le processus Python automatiquement à chaque modification — un mécanisme
  incompatible avec un kernel Jupyter, qu'il interromprait de façon incontrôlée.
- `threaded=True` : **indispensable dès qu'il y a plus d'une caméra**. Sans ce paramètre, le serveur de
  développement ne traite qu'une seule requête à la fois. Or chaque flux vidéo (`/video_feed/<camera>`) est une
  réponse *infinie* (le générateur `generer_frames()` ne se termine jamais tant que la caméra est branchée) — la
  première caméra qui se connecte monopoliserait alors indéfiniment l'unique thread disponible, et toute autre
  requête (une deuxième caméra, ou même une simple navigation) resterait bloquée en attente.

**Origine — WSGI** : le serveur intégré de Flask parle le protocole **WSGI** (*Web Server Gateway Interface*,
normalisé par la *PEP 3333* en 2010), qui définit une interface standard entre un serveur web et une application
Python — n'importe quelle application respectant WSGI peut tourner derrière n'importe quel serveur compatible.
WSGI a succédé au protocole **CGI** (*Common Gateway Interface*, créé en 1993 par le NCSA, l'un des tout premiers
standards ayant permis à des scripts serveur de générer des pages web dynamiques). Ce serveur de développement
n'est volontairement pas conçu pour un usage en production à forte charge — Flask le rappelle d'ailleurs dans ses
propres avertissements de démarrage ; des serveurs comme Gunicorn ou uWSGI prendraient le relais pour un déploiement
réel, mais restent hors du périmètre de ce projet académique.

In [48]:
demarrer_cameras()
app.run(host="0.0.0.0", port=5000, debug=True, use_reloader=False, threaded=True)

Thread de détection démarré pour CAM-01.
Thread de détection démarré pour CAM-02.
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.100.95:5000
Press CTRL+C to quit
192.168.100.95 - - [02/Sep/2026 09:19:52] "GET / HTTP/1.1" 200 -
192.168.100.95 - - [02/Sep/2026 09:19:52] "GET / HTTP/1.1" 200 -
192.168.100.95 - - [02/Sep/2026 09:19:53] "GET /static/style.css HTTP/1.1" 304 -
192.168.100.95 - - [02/Sep/2026 09:19:53] "GET /static/icones%20images/logo_inch.jpg HTTP/1.1" 304 -
192.168.100.95 - - [02/Sep/2026 09:19:53] "GET /static/style.css HTTP/1.1" 304 -
192.168.100.95 - - [02/Sep/2026 09:19:53] "GET /static/icones%20images/logo_inch.jpg HTTP/1.1" 304 -
192.168.100.95 - - [02/Sep/2026 09:19:54] "GET /favicon.ico HTTP/1.1" 404 -
192.168.100.95 - - [02/Sep/2026 09:19:57] "GET /choix_langue/en HTTP/1.1" 302 -
192.168.100.95 - - [02/Sep/2026 09:19:57] "GET / HTTP/1.1" 200 -
192.168.100.95 - - [02/Sep/2026 09:19:57] "GET /static/style.css HTTP/1.1" 304 -
192.168.100.95 - - [02/Sep/2026 09:19:57] "GET /static/icones%20imag